# 35 · CXR-LT Stage 4 — sàng lọc rẻ trên embedding **đã cache**

Bốn phép đo, **không trích lại embedding**, không đụng test.

| | nội dung | cổng khoá trước |
|---|---|---|
| **A2** | `Case C` — `gain_cls` vs `gain_patch`: EHR mang thông tin độc lập hay chỉ vá bottleneck của pooling | không có cổng — đây là phép **chẩn đoán**, đọc dấu và độ lớn |
| **A1** | ASL / WASL thay BCE | `ASL − BCE ≥ +0.003` mAP ảnh |
| **B1** | đầu đọc **phi tuyến** + ba nhánh missingness | `MLP − linear ≥ +0.003` |
| **A5.1** | mốc tương quan giữa 9 `p_*.npy` đã có + lợi ích ensemble | không có cổng — là **mốc đối chiếu** cho `41a` |

**Kỷ luật.**

* Mọi lựa chọn (epoch dừng, siêu tham số) làm trên **lát cắt 15% cắt từ TRAIN theo `subject_id`**, không bao giờ trên val. Val chỉ dùng để **báo cáo**.
* Early-stop theo **`tail mAP`**, không theo macro — 11 nhãn tail là toàn bộ khoảng cách 3.26× tới vô địch, macro bị 13 nhãn head chi phối.
* Sàn nhiễu **0.005**: chênh lệch dưới mức đó không đọc là thắng.
* `ASL` / `WASL` / `class_weights` lấy **verbatim** từ `12`/`15`, có cổng bit-for-bit — để `ASL − BCE` ở đây so được với `+0.0050` (p=0.029) của cohort cũ.

## C0 · Config, nạp dữ liệu, cắt lát chọn mẫu 85/15

In [1]:
import os, json, time, itertools, math, re
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RES_DIR = os.path.join(OUT_DIR, 'stage3_baselines')                       # nb32 ghi o day
RES35   = os.path.join(OUT_DIR, 'stage4_nb35'); os.makedirs(RES35, exist_ok=True)

W_FRONTAL, W_LATERAL = 0.7, 0.3
SEED      = 42
SEL_FRAC  = 0.15          # lat cat chon mau, cat tu TRAIN theo subject_id
NOISE     = 0.005         # san nhieu — hang 2..4 cua giai cach nhau 0.004
EPOCHS    = 60            # ngan sach CHUNG cho moi dau doc. Lat chon tu chon epoch dung.
                          # Phai du rong cho ca linear (hoi tu ~ep20) lan MLP (cham hon han):
                          # neu mot arm dung dung o EPOCHS thi phep so bien thanh so NGAN SACH.
ARM, KIND = 'nosq518', 'patch'                                            # arm chuan cua Stage 3

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']
TIER  = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head', 'medium', 'tail')}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')

P_VIEW = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
if os.path.exists(P_VIEW):
    V = (pd.read_parquet(P_VIEW)[['dicom_id', 'view_group', 'use']]
           .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
else:
    V = ORDER[['dicom_id', 'view_group']].copy(); V['use'] = V.view_group != 'unknown'

SPLIT = ORDER.split.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'

# --- lat cat chon mau: cat theo SUBJECT, khong theo anh (mot benh nhan nhieu study/anh) ---
SUB = ORDER.subject_id.to_numpy()
_rs   = np.random.default_rng(SEED)
_subs = np.unique(SUB[TR])
_sel  = _rs.choice(_subs, size=int(round(SEL_FRAC * len(_subs))), replace=False)
SL  = TR & np.isin(SUB, _sel)          # 15% — CHON tren day
TF  = TR & ~SL                         # 85% — KHOP tren day
assert not (set(SUB[SL]) & set(SUB[TF])), 'subject tran giua hai lat cat train'
assert not (set(SUB[TR]) & set(SUB[VA])), 'subject tran giua train va val'

_npos_sl = YM[SL].sum(0)
assert _npos_sl.min() >= 5, f'nhan hiem nhat chi co {_npos_sl.min():.0f} duong trong lat chon -> tang SEL_FRAC'

ARMS_HAVE = [a for a in ('crop518', 'lb518', 'nosq518', 'lb1022')
             if os.path.exists(os.path.join(EMB_DIR, f'emb_{a}_cls.npy'))]
assert ARM in ARMS_HAVE, f'thieu embedding {ARM} -> chay 31'

print(f'{len(ORDER):,} anh | train khop {TF.sum():,} | train chon {SL.sum():,} | val {VA.sum():,}')
print(f'arm da co: {ARMS_HAVE}   |   arm chuan: {ARM}/{KIND}')
print(f'lat chon: {len(_sel):,}/{len(_subs):,} subject | nhan hiem nhat co {_npos_sl.min():.0f} duong')
print(f'tier: head {len(TIER["head"])} | medium {len(TIER["medium"])} | tail {len(TIER["tail"])}')

Mounted at /content/drive
377,110 anh | train khop 219,785 | train chon 39,086 | val 39,293
arm da co: ['crop518', 'lb518', 'nosq518']   |   arm chuan: nosq518/patch
lat chon: 6,822/45,480 subject | nhan hiem nhat co 20 duong
tier: head 13 | medium 16 | tail 11


## C1 · Hàm mất mát — **verbatim `12`/`15`**, có cổng bit-for-bit

`ASL` ở đây **không** phải bản tôi tự viết lại. Nó là lớp `ASLArm` của `15` (chính là `ASLoss` của `12`,
cùng `gn=4, gp=1, clip=0.05`), và `WASL` là `WeightedASL` với trọng số liên lớp **chỉ trên số hạng dương**.
Hai `assert` dưới bắt buộc: `ASL` khớp tham chiếu, và `WASL(w=1)` **bằng đúng** `ASL` — nếu không thì
`Δ_WASL` sẽ lẫn cả sai lệch cài đặt.

`class_weights`: `w = clip(s · n^{-α}, 0.5, 3.0)`, `s` tìm bằng bisection sao cho `mean(w) = 1`.
Chuẩn hoá về trung bình 1 để **độ lớn loss không đổi** — nếu không, đổi `α` cũng là đổi learning rate hiệu dụng.

In [2]:
import torch, torch.nn as nn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'thiet bi: {dev}')

class ASL(nn.Module):
    '''ASL cua nb11/12/13/15 — bit-for-bit.'''
    def __init__(s, gn=4, gp=1, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()

class WASL(nn.Module):
    '''ASL + trong so lien lop CHI tren so hang duong. w=1 -> identical ASL. Verbatim nb12.'''
    def __init__(s, w, gn=4, gp=1, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
        s.register_buffer('w', torch.as_tensor(np.asarray(w, 'float32')))
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lp = y*torch.log(xp.clamp(min=s.eps)); ln = (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y); mod = torch.pow(1-pt, g)
        return (-(s.w*lp)*mod - ln*mod).mean()

def _asl_ref(x, y, gn=4, gp=1, clip=0.05, eps=1e-8):
    y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
    if clip: xn = (xn + clip).clamp(max=1)
    lo = y*torch.log(xp.clamp(min=eps)) + (1-y)*torch.log(xn.clamp(min=eps))
    pt = xp*y + xn*(1-y); g = gp*y + gn*(1-y)
    return -(lo * torch.pow(1-pt, g)).mean()

torch.manual_seed(0)
_x = torch.randn(64, len(LAB40), device=dev)
_y = (torch.rand(64, len(LAB40), device=dev) < 0.15).float()
assert torch.equal(ASL().to(dev)(_x, _y), _asl_ref(_x, _y)), 'ASL lech so voi nb11/12/13/15'
assert torch.equal(WASL(np.ones(len(LAB40))).to(dev)(_x, _y), ASL().to(dev)(_x, _y)), 'WASL(w=1) != ASL'
print('cong bit-for-bit: ASL khop nb12/15  |  WASL(w=1) == ASL   OK')

def class_weights(n_pos, alpha, lo=0.5, hi=3.0, tol=1e-12, iters=200):
    '''w = clip(s * n^-alpha, lo, hi), s bisection de mean(w)=1. Verbatim nb12/15.'''
    n = np.asarray(n_pos, 'float64'); assert np.all(n > 0)
    if alpha == 0: return np.ones(len(n))
    x = n ** (-alpha)
    f = lambda s: np.clip(s*x, lo, hi).mean() - 1.0
    s_lo, s_hi = 1e-12, 1.0
    while f(s_hi) < 0: s_hi *= 2.0
    for _ in range(iters):
        s = 0.5*(s_lo + s_hi)
        if f(s) < 0: s_lo = s
        else:        s_hi = s
        if s_hi - s_lo < tol*max(1.0, s_hi): break
    return np.clip(0.5*(s_lo+s_hi)*x, lo, hi)

N_POS_TF = YM[TF].sum(0)
W50 = class_weights(N_POS_TF, 0.50)
assert abs(W50.mean() - 1) < 1e-6 and W50.min() >= 0.5 - 1e-9 and W50.max() <= 3.0 + 1e-9
print(f'WASL alpha=.50 | mean {W50.mean():.8f} | min {W50.min():.3f} | max {W50.max():.3f} | '
      f'cham tran {(W50 >= 3.0-1e-9).sum()}/40 | cham san {(W50 <= 0.5+1e-9).sum()}/40')
print(f'  n_pos train khop: {N_POS_TF.min():.0f} .. {N_POS_TF.max():.0f}  ({N_POS_TF.max()/N_POS_TF.min():.0f}x)')

thiet bi: cuda
cong bit-for-bit: ASL khop nb12/15  |  WASL(w=1) == ASL   OK
WASL alpha=.50 | mean 1.00000000 | min 0.500 | max 3.000 | cham tran 1/40 | cham san 15/40
  n_pos train khop: 109 .. 73311  (673x)


## C2 · AP nhanh (tie-aware) + máy huấn luyện chung

`ap_cols` gộp các điểm **bằng nhau** thành **một ngưỡng** — nếu tách theo thứ tự sort thì AP lệch khỏi
`sklearn` (đã dính ở `33` C4, lệch `2.8e-07`). C7 sẽ đối chiếu lại với `sklearn` để chứng minh.

`fit_head` là một hàm duy nhất cho **cả bốn** thí nghiệm: đổi `head` (linear/mlp), đổi `loss`
(bce/asl/wasl), và **luôn** early-stop theo `tail mAP` trên lát cắt 15% của train.
Chuẩn hoá `mean/std` lấy **chỉ từ lát khớp** — không chạm lát chọn, không chạm val.

In [3]:
def ap_cols(Ycol, Scol):
    '''AP tung cot, gop diem BANG NHAU thanh mot nguong (khop sklearn).'''
    out = np.full(Ycol.shape[1], np.nan)
    for j in range(Ycol.shape[1]):
        y, s = Ycol[:, j], Scol[:, j]
        n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        o = np.argsort(-s, kind='stable'); yy = y[o]; ss = s[o]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]
        tp, fp = tp[last], fp[last]
        prec, rec = tp / (tp + fp), tp / n1
        out[j] = float((np.diff(np.r_[0.0, rec]) * prec).sum())
    return out

tiers = lambda ap: {t: float(np.nanmean(ap[ix])) for t, ix in TIER.items()}

def canh_bao_tran(df, col='ep'):
    '''ep* == EPOCHS nghia la tail VAN DANG LEN khi het ngan sach -> arm do bi NGAN SACH chan,
       khong phai bi kien truc chan. So hai arm trong tinh trang do la so ngan sach.'''
    hit = [i for i in df.index if int(df.loc[i, col]) >= EPOCHS]
    if hit:
        print()
        print(f'  !! {len(hit)} arm dung dung o tran {EPOCHS} epoch: {hit}')
        print('     -> tail con dang len. Tang EPOCHS roi chay lai truoc khi doc bat ky chenh lech nao.')
    return bool(hit)

class MLPHead(nn.Module):
    def __init__(s, d, c, hid=1024, drop=0.2):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(d, hid), nn.GELU(), nn.Dropout(drop), nn.Linear(hid, c))
    def forward(s, x): return s.net(x)

def fit_head(X, y, *, head='linear', loss='bce', epochs=None, lr=3e-3, wd=1e-4, bs=4096,
             hid=None, drop=0.2, alpha=0.50, seed=SEED, verbose=False):
    '''Khop tren TF, chon epoch tren SL (tail mAP), tra du doan tren VA.
       X: [N, D] toan bo anh (dung mask TF/SL/VA ben trong).'''
    epochs = EPOCHS if epochs is None else epochs
    torch.manual_seed(seed)
    mu, sd = X[TF].mean(0, keepdims=True), X[TF].std(0, keepdims=True) + 1e-6   # CHI lat khop
    z  = lambda m: torch.from_numpy(((X[m] - mu) / sd).astype('float32')).to(dev)
    xt, xs, xv = z(TF), z(SL), z(VA)
    yt = torch.from_numpy(y[TF]).to(dev)
    ys = y[SL]

    hid = min(1024, 4*xt.shape[1]) if hid is None else hid
    m = (nn.Linear(xt.shape[1], y.shape[1]) if head == 'linear'
         else MLPHead(xt.shape[1], y.shape[1], hid, drop)).to(dev)
    crit = {'bce':  nn.BCEWithLogitsLoss(),
            'asl':  ASL(),
            'asl20': ASL(gn=2, gp=0), 'asl40': ASL(gn=4, gp=0), 'asl41': ASL(gn=4, gp=1),
            'wasl': WASL(class_weights(N_POS_TF, alpha))}[loss].to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)

    # bs co dinh 4096 tren tap nho -> chi vai buoc optimizer moi epoch, MLP khong the hoi tu va
    # `ep*` se luon cham tran. Dat SAN 64 buoc/epoch. Tren du lieu THAT dieu nay khong doi gi:
    # 220,000 / 64 = 3,437 > 4096 nen bs van bang 4096.
    bs = int(min(bs, max(256, len(xt) // 64)))
    n, best, best_ep, curve = len(xt), -1.0, -1, []
    p_best = None
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(n, device=dev)
        for i in range(0, n, bs):
            ix = perm[i:i+bs]
            opt.zero_grad(set_to_none=True)
            crit(m(xt[ix]), yt[ix]).backward(); opt.step()
        sch.step()
        m.eval()
        with torch.no_grad():
            ps = torch.sigmoid(m(xs)).cpu().numpy()
        ap = ap_cols(ys, ps); tk = tiers(ap)
        curve.append(dict(ep=ep+1, macro=float(np.nanmean(ap)), **tk))
        if tk['tail'] > best:                       # DUNG theo tail, khong theo macro
            best, best_ep = tk['tail'], ep + 1
            with torch.no_grad():
                p_best = torch.sigmoid(m(xv)).cpu().numpy()
        if verbose and (ep+1) % 10 == 0:
            print(f'   ep{ep+1:>3} sel macro {curve[-1]["macro"]:.4f} tail {tk["tail"]:.4f}')
    del xt, xs, xv, yt
    if dev == 'cuda': torch.cuda.empty_cache()
    return p_best.astype('float32'), best_ep, pd.DataFrame(curve)

def feats(arm, kind):
    '''Nap HAN vao RAM, khong tra ve view cua memmap.

    `32` dung np.asarray(np.load(..., mmap_mode="r")) — voi dtype khop thi do la VIEW chu khong
    phai ban sao, nen moi lan `X[TF]` la mot lan doc lai tu Drive. `32` chi khop 1 lan moi to hop
    nen chiu duoc; `35` khop 17 lan tren CUNG mot mang => 17 x 1.16 GB doc qua Drive (~83 MB/s)
    la ~4 phut MOI LAN. np.array(...) doc mot lan roi thoi.  patch 1.16 GB | both 2.3 GB.'''
    c = lambda n: np.array(np.load(os.path.join(EMB_DIR, f'emb_{arm}_{n}.npy'), mmap_mode='r'))
    if kind in ('cls', 'patch'): return c(kind)
    return np.concatenate([c('cls'), c('patch')], 1)
print('san sang')

san sang


## C3 · Chấm điểm hai mức — sao chép nguyên `32` C2

Giữ nguyên để mọi con số ở đây **cùng thang** với `cxr_only.csv` của `32`.

In [4]:
from sklearn.metrics import average_precision_score

_sid_va = ORDER.study_id.to_numpy()[VA]
_uid, _inv = np.unique(_sid_va, return_inverse=True)
_isF = (V.view_group.to_numpy()[VA] == 'frontal') & V.use.to_numpy()[VA]
_isL = (V.view_group.to_numpy()[VA] == 'lateral') & V.use.to_numpy()[VA]

def pool_study(p_img):
    C = p_img.shape[1]; out = np.zeros((len(_uid), C), 'float64'); wsum = np.zeros(len(_uid))
    for mask, w in ((_isF, W_FRONTAL), (_isL, W_LATERAL)):
        if not mask.any(): continue
        cnt = np.bincount(_inv[mask], minlength=len(_uid))
        acc = np.stack([np.bincount(_inv[mask], weights=p_img[mask, j].astype('float64'),
                                    minlength=len(_uid)) for j in range(C)], 1)
        has = cnt > 0
        out[has] += w * acc[has] / cnt[has, None]; wsum[has] += w
    ok = wsum > 0
    out[ok] /= wsum[ok, None]
    return out, ok

_first = np.zeros(len(_uid), int); _first[_inv[::-1]] = np.arange(VA.sum())[::-1]
_Y_study = YM[VA][_first]

def score(p_img):
    ya = YM[VA]
    ap_i = np.array([average_precision_score(ya[:, j], p_img[:, j]) for j in range(ya.shape[1])])
    ps, ok = pool_study(p_img)
    ap_s = np.array([average_precision_score(_Y_study[ok][:, j], ps[ok][:, j])
                     for j in range(ya.shape[1])])
    return ap_i, ap_s

def row_of(nm, p, **extra):
    ai, asx = score(p)
    return dict(ten=nm, mAP_anh=float(ai.mean()), mAP_study=float(asx.mean()),
                **tiers(ai), **extra, **{f'ap_{l}': v for l, v in zip(LAB40, ai)})

print(f'val: {VA.sum():,} anh -> {len(_uid):,} study | frontal {_isF.sum():,} | lateral {_isL.sum():,}')
print(f'ti le study val CO lateral: {(np.bincount(_inv[_isL], minlength=len(_uid)) > 0).mean():.1%}'
      f'   <- doi chieu 48.3% cua cohort cu (A6)')

val: 39,293 anh -> 23,800 study | frontal 26,456 | lateral 12,720
ti le study val CO lateral: 48.8%   <- doi chieu 48.3% cua cohort cu (A6)


## C4 · Nạp EHR — ba nhánh missingness

`29b` giữ NaN và thêm cột `*_mask`; `32` C6 làm `nan_to_num(0)` **trước** chuẩn hoá, nên "thiếu" và
"bằng 0 thô" trùng nhau. Ba nhánh tách bạch ba giả thuyết:

```
zero          nan -> 0 TRUOC chuan hoa      = verbatim 32 C6 (de so duoc)
median_mask   nan -> trung vi cua LAT KHOP  + cot *_mask (da co san trong Q_FULL)
mask_only     Q_CTX — bo HET gia tri lam sang, chi giu missingness + dem + boi canh
```

`mask_only` là phép thử mạnh nhất: nếu nó gần bằng `zero` thì EHR chủ yếu đang đọc ra *loại bệnh nhân*
(nội trú / ED / ngoại trú), không đọc ra *bệnh*.

In [5]:
P_EHR = os.path.join(OUT_DIR, 'cxrlt_ehr.parquet')
HAVE_EHR = os.path.exists(P_EHR) and os.path.exists(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json'))
EHR = {}
if not HAVE_EHR:
    print('!! chua co cxrlt_ehr.parquet -> A2 va B1 se bo qua nhanh EHR (chay 29b truoc)')
else:
    SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
    E = pd.read_parquet(P_EHR)
    _sid = ORDER.study_id.to_numpy()

    def _mat(cols):
        return (E.set_index('study_id')[cols].reindex(_sid)
                 .apply(pd.to_numeric, errors='coerce').to_numpy('float32'))

    C_FULL = [c for c in SPEC['Q_FULL'] if c in E.columns]
    C_CTX  = [c for c in SPEC['Q_CTX']  if c in E.columns]
    RAW = _mat(C_FULL)

    EHR['zero'] = np.nan_to_num(RAW, nan=0.0)                        # verbatim 32 C6

    # trung vi CHI tren lat khop. Lay tren ANH (khong phai study) vi do dung la phan phoi ma
    # mo hinh nhin thay; study nhieu anh dung la duoc dem nhieu lan.
    with np.errstate(all='ignore'):
        _med = np.nanmedian(RAW[TF], axis=0)      # cot toan NaN -> NaN, ha xuong 0 ngay duoi
    _med = np.nan_to_num(_med, nan=0.0)
    EHR['median_mask'] = np.where(np.isnan(RAW), _med, RAW).astype('float32')

    EHR['mask_only'] = np.nan_to_num(_mat(C_CTX), nan=0.0)

    _nan = np.isnan(RAW)
    print(f'EHR: Q_FULL {len(C_FULL)} cot | Q_CTX {len(C_CTX)} cot | phat cho {len(RAW):,} anh')
    print(f'  o thieu: {_nan.mean():.1%} | anh khong co MOT gia tri nao: {(_nan.all(1)).sum():,}')
    print(f'  cot *_mask trong Q_FULL: {sum(c.endswith("_mask") for c in C_FULL)}')
    for k, v in EHR.items():
        assert np.isfinite(v).all(), f'{k} con NaN/inf'
        print(f'  {k:<12} {v.shape[1]:>4} cot')

EHR: Q_FULL 247 cot | Q_CTX 157 cot | phat cho 377,110 anh
  o thieu: 22.6% | anh khong co MOT gia tri nao: 0
  cot *_mask trong Q_FULL: 45
  zero          247 cot
  median_mask   247 cot
  mask_only     157 cot


## C5 · **A2 · Case C** — EHR mang thông tin độc lập, hay chỉ vá bottleneck của pooling?

Cohort cũ: EHR trên `cls` cho `+0.0081` nhưng trên `patch` chỉ `+0.0003` ⇒ phần lớn gain của EHR chỉ là
**bù cho việc global pooling vứt mất thông tin không gian**. Ta đã có `+0.0040` trên `patch` của CXR-LT
nhưng **chưa có** số trên `cls`, nên chưa biết kết luận đó có lặp lại trên 40 nhãn không.

```
gain_cls   = EHR+CXR(cls)   - CXR-only(cls)
gain_patch = EHR+CXR(patch) - CXR-only(patch)
```

* `gain_cls >> gain_patch` ⇒ **Case C lặp lại** — EHR chủ yếu vá pooling, và ML-Decoder ở `37` sẽ nuốt mất phần lớn gain đó.
* `gain_cls ≈ gain_patch` ⇒ trên 40 nhãn EHR mang thông tin **độc lập** — **khác** cohort cũ, và đó tự nó là phát hiện.

Bốn lần khớp, cùng loss BCE, cùng đầu đọc tuyến tính — **chỉ đổi biểu diễn và có/không EHR**.

In [6]:
P_A2 = os.path.join(RES35, 'a2_case_c.csv')
rows = [] if not os.path.exists(P_A2) else pd.read_csv(P_A2).to_dict('records')
_done = {r['ten'] for r in rows}

if HAVE_EHR:
    for kd in ('cls', 'patch'):
        X = feats(ARM, kd)
        for tag, Xc in (('cxr', X), ('cxr_ehr', np.concatenate([X, EHR['zero']], 1))):
            nm = f'{kd}_{tag}'
            if nm in _done: continue
            t0 = time.time()
            p, ep, cur = fit_head(Xc, YM, head='linear', loss='bce')
            np.save(os.path.join(RES35, f'p_a2_{nm}.npy'), p)
            rows.append(row_of(nm, p, ep=ep, D=Xc.shape[1]))
            pd.DataFrame(rows).to_csv(P_A2, index=False)
            print(f'  {nm:<14} mAP anh {rows[-1]["mAP_anh"]:.4f} | tail {rows[-1]["tail"]:.4f} '
                  f'| ep* {ep:>2} | {time.time()-t0:.0f}s')
            del Xc
        del X

# khong co EHR -> `rows` rong -> DataFrame khong co cot 'ten' -> drop_duplicates nem KeyError.
A2 = (pd.DataFrame(rows).drop_duplicates('ten', keep='last').set_index('ten') if rows
      else pd.DataFrame())
if len(A2) == 4:
    g_cls   = A2.loc['cls_cxr_ehr',   'mAP_anh'] - A2.loc['cls_cxr',   'mAP_anh']
    g_patch = A2.loc['patch_cxr_ehr', 'mAP_anh'] - A2.loc['patch_cxr', 'mAP_anh']
    show = ['mAP_anh', 'mAP_study', 'head', 'medium', 'tail', 'ep', 'D']
    print('\n=== A2 · CASE C ===')
    print(A2[show].to_string(float_format=lambda v: f'{v:.4f}'))
    canh_bao_tran(A2)
    b_cls, b_patch = A2.loc['cls_cxr', 'mAP_anh'], A2.loc['patch_cxr', 'mAP_anh']
    _dif = g_cls - g_patch
    _r   = g_patch / g_cls if abs(g_cls) > 1e-9 else np.inf
    print(f'\n  nen (CXR-only)  cls {b_cls:.4f} | patch {b_patch:.4f}')
    print(f'  gain tuyet doi  cls {g_cls:+.4f} | patch {g_patch:+.4f}   (32 C6 do +0.0040 @patch)')
    print(f'  gain / nen      cls {g_cls/b_cls:+.2%} | patch {g_patch/b_patch:+.2%}')
    print(f'  hieu            {_dif:+.4f}')
    print(f'  TI SO g_patch/g_cls = {_r:+.3f}   <- so quyet dinh')
    # Doc bang TI SO chu khong bang HIEU: `patch` co nen cao hon `cls` nen cung mot luong thong tin
    # them vao se cho delta-AP TUYET DOI lon hon o `patch`. Hieu tuyet doi vi the tron lan
    # "EHR doc lap" voi "hai nen khac nhau"; ti so thi khong.
    print(f'\n  ket doc: ', end='')
    if _dif > NOISE and _r < 0.5:
        print(f'CASE C LAP LAI — g_cls lon ({g_cls:+.4f}) trong khi g_patch gan 0 ({g_patch:+.4f}).')
        print('           `patch` DA CO san thu ma EHR mang lai => EHR chu yeu va bottleneck pooling.')
        print('           => ky vong gain EHR SUT khi len ML-Decoder o 37; REPORT phai noi ro dieu do.')
    elif _r >= 0.5:
        print(f'KHONG lap lai — g_patch bang {_r:.0%} cua g_cls, khong he gan 0.')
        print('           Tren 40 nhan EHR mang thong tin DOC LAP voi pooling.')
        print('           Khac cohort cu; day la PHAT HIEN, ghi vao REPORT.')
    else:
        print(f'KHONG KET LUAN — g_cls {g_cls:+.4f} qua nho de lay ti so; ca hai deu quanh san nhieu.')
    # phan ra tung nhan: EHR va pooling co cung mot nhom nhan khong
    _lc = np.array([A2.loc['cls_cxr_ehr',   f'ap_{l}'] - A2.loc['cls_cxr',   f'ap_{l}'] for l in LAB40])
    _lp = np.array([A2.loc['patch_cxr_ehr', f'ap_{l}'] - A2.loc['patch_cxr', f'ap_{l}'] for l in LAB40])
    _pool = np.array([A2.loc['patch_cxr', f'ap_{l}'] - A2.loc['cls_cxr', f'ap_{l}'] for l in LAB40])
    print(f'\n  corr(lift EHR@cls, loi ich cua patch) = {np.corrcoef(_lc, _pool)[0,1]:+.3f}')
    print('    duong manh => dung nhung nhan ma pooling lam hong la nhung nhan EHR cuu -> Case C')
    TB = pd.DataFrame(dict(nhan=LAB40, tier=[t for l in LAB40 for t in TIER
                                             if LAB40.index(l) in TIER[t]],
                           lift_cls=_lc, lift_patch=_lp, loi_patch=_pool))
    print('\n  10 nhan EHR@cls nhac manh nhat:')
    print(TB.sort_values('lift_cls', ascending=False).head(10)
            .to_string(index=False, float_format=lambda v: f'{v:+.4f}'))


=== A2 · CASE C ===
               mAP_anh  mAP_study   head  medium   tail  ep     D
ten                                                              
cls_cxr         0.1713     0.1836 0.3995  0.0945 0.0131  57   768
cls_cxr_ehr     0.1746     0.1864 0.4065  0.0970 0.0134  54  1015
patch_cxr       0.2011     0.2121 0.4182  0.1345 0.0412  49   768
patch_cxr_ehr   0.2048     0.2161 0.4250  0.1383 0.0414  49  1015

  nen (CXR-only)  cls 0.1713 | patch 0.2011
  gain tuyet doi  cls +0.0034 | patch +0.0038   (32 C6 do +0.0040 @patch)
  gain / nen      cls +1.96% | patch +1.87%
  hieu            -0.0004
  TI SO g_patch/g_cls = +1.120   <- so quyet dinh

  ket doc: KHONG lap lai — g_patch bang 112% cua g_cls, khong he gan 0.
           Tren 40 nhan EHR mang thong tin DOC LAP voi pooling.
           Khac cohort cu; day la PHAT HIEN, ghi vao REPORT.

  corr(lift EHR@cls, loi ich cua patch) = -0.261
    duong manh => dung nhung nhan ma pooling lam hong la nhung nhan EHR cuu -> Case C

  10 nhan

## C6 · **A1 · ASL / WASL thay BCE**

Cohort cũ (13 nhãn): **ASL − BCE = +0.0050** (p=0.029), thắng cả ba tier.
Cổng ở đây đặt **+0.003** — nửa hiệu ứng cũ, vì 40 nhãn ≠ 13 nhãn.

**Bẫy đã biết:** WASL α=.50 trên cohort cũ **overfit rất nhanh** — tail `0.1230@ep2 → 0.1098@ep8`
(`−0.0132`). Đó là lý do `fit_head` early-stop theo `tail mAP` trên lát chọn. Cột `ep*` cho biết
epoch được chọn; `ep*` rất nhỏ ở WASL là **triệu chứng**, không phải lỗi.

In [7]:
P_A1 = os.path.join(RES35, 'a1_loss.csv')
rows = [] if not os.path.exists(P_A1) else pd.read_csv(P_A1).to_dict('records')
_done = {r['ten'] for r in rows}
CURVE = {}

LOSSES = [('bce',   'BCE (moc goc)'),
          ('asl20', 'ASL gn=2 gp=0'),
          ('asl40', 'ASL gn=4 gp=0'),
          ('asl41', 'ASL gn=4 gp=1  <- cau hinh nb11/12/13'),
          ('wasl',  'WASL alpha=.50')]

X = feats(ARM, KIND)
for key, lab in LOSSES:
    if key in _done: continue
    t0 = time.time()
    p, ep, cur = fit_head(X, YM, head='linear', loss=key)
    np.save(os.path.join(RES35, f'p_a1_{key}.npy'), p)
    cur.to_csv(os.path.join(RES35, f'curve_a1_{key}.csv'), index=False)
    rows.append(row_of(key, p, ep=ep, mo_ta=lab))
    pd.DataFrame(rows).to_csv(P_A1, index=False)
    print(f'  {key:<6} mAP anh {rows[-1]["mAP_anh"]:.4f} | tail {rows[-1]["tail"]:.4f} '
          f'| ep* {ep:>2}/30 | {time.time()-t0:.0f}s')
del X

A1 = pd.DataFrame(rows).drop_duplicates('ten', keep='last').set_index('ten')
for key, _ in LOSSES:
    f = os.path.join(RES35, f'curve_a1_{key}.csv')
    if os.path.exists(f): CURVE[key] = pd.read_csv(f)

show = ['mo_ta', 'mAP_anh', 'mAP_study', 'head', 'medium', 'tail', 'ep']
print('\n=== A1 · LOSS ===')
print(A1[show].to_string(float_format=lambda v: f'{v:.4f}'))
canh_bao_tran(A1)

base = A1.loc['bce', 'mAP_anh']
print(f'\n  chenh so voi BCE (san nhieu {NOISE}, cong {"+0.003"}):')
for key, _ in LOSSES:
    if key == 'bce' or key not in A1.index: continue
    d  = A1.loc[key, 'mAP_anh'] - base
    dt = A1.loc[key, 'tail']    - A1.loc['bce', 'tail']
    print(f'    {key:<6} anh {d:+.4f} | tail {dt:+.4f}   '
          f'{"QUA CONG" if d >= 0.003 else ("trong san" if abs(d) < NOISE else "duoi cong")}')
print(f'\n  cohort cu (13 nhan): ASL - BCE = +0.0050 (p=0.029)')

# trieu chung overfit: tail tren lat chon di xuong bao nhieu tu dinh den ep cuoi
print('\n  duong cong TAIL tren lat chon (dinh -> ep30):')
for key, _ in LOSSES:
    if key not in CURVE: continue
    c = CURVE[key]; i = int(c['tail'].idxmax())
    print(f'    {key:<6} dinh {c["tail"].iloc[i]:.4f}@ep{int(c.ep.iloc[i]):>2} -> '
          f'ep{len(c)} {c["tail"].iloc[-1]:.4f}  ({c["tail"].iloc[-1]-c["tail"].iloc[i]:+.4f})')


=== A1 · LOSS ===
                                       mo_ta  mAP_anh  mAP_study   head  medium   tail  ep
ten                                                                                       
bce                            BCE (moc goc)   0.2011     0.2121 0.4182  0.1345 0.0412  49
asl20                          ASL gn=2 gp=0   0.2061     0.2197 0.4208  0.1327 0.0594  60
asl40                          ASL gn=4 gp=0   0.2083     0.2225 0.4195  0.1377 0.0614  49
asl41  ASL gn=4 gp=1  <- cau hinh nb11/12/13   0.2078     0.2214 0.4199  0.1368 0.0603  55
wasl                          WASL alpha=.50   0.2079     0.2214 0.4200  0.1369 0.0605  55

  !! 1 arm dung dung o tran 60 epoch: ['asl20']
     -> tail con dang len. Tang EPOCHS roi chay lai truoc khi doc bat ky chenh lech nao.

  chenh so voi BCE (san nhieu 0.005, cong +0.003):
    asl20  anh +0.0051 | tail +0.0182   QUA CONG
    asl40  anh +0.0072 | tail +0.0202   QUA CONG
    asl41  anh +0.0067 | tail +0.0191   QUA CONG
    was

## C7 · **B1 · Đầu đọc phi tuyến** + ba nhánh missingness

Theo Huang et al. (npj Digit Med 2020), `32` C6 là **Early Fusion Type II**. Bảng 2 ghi công cho early
fusion ở chỗ *"model interactions between features from different modalities"* — nhưng `32` C6 đặt lên
trên một **lớp tuyến tính** `W_e·e + W_h·h`, **không một số hạng chéo nào**. Ta đã ném đi đúng khả năng đó.

Và nó là **cùng một vấn đề** với missingness: để phân biệt "thiếu" khỏi "bằng trung bình", mô hình phải
học tích `mask × value` — **lớp tuyến tính không nhân được hai đặc trưng**.

Prior làm nguội kỳ vọng: `06` đo **FiLM ≈ concat, p = 0.308** ở mức CLS ⇒ *cơ chế* fusion không quan
trọng ở mức đó. Cái đáng kỳ vọng ở đây là **phi tuyến**, vì mốc của ta hiện là tuyến tính còn `06` đã là MLP.

In [8]:
P_B1 = os.path.join(RES35, 'b1_head.csv')
rows = [] if not os.path.exists(P_B1) else pd.read_csv(P_B1).to_dict('records')
_done = {r['ten'] for r in rows}

X = feats(ARM, KIND)
JOBS = [('cxr_linear', 'linear', None), ('cxr_mlp', 'mlp', None)]
if HAVE_EHR:
    for br in ('zero', 'median_mask', 'mask_only'):
        JOBS += [(f'{br}_linear', 'linear', br), (f'{br}_mlp', 'mlp', br)]

for nm, hd, br in JOBS:
    if nm in _done: continue
    Xc = X if br is None else np.concatenate([X, EHR[br]], 1)
    t0 = time.time()
    p, ep, cur = fit_head(Xc, YM, head=hd, loss='bce')
    np.save(os.path.join(RES35, f'p_b1_{nm}.npy'), p)
    rows.append(row_of(nm, p, ep=ep, dau_doc=hd, nhanh=br or 'khong_ehr', D=Xc.shape[1]))
    pd.DataFrame(rows).to_csv(P_B1, index=False)
    print(f'  {nm:<18} mAP anh {rows[-1]["mAP_anh"]:.4f} | tail {rows[-1]["tail"]:.4f} '
          f'| ep* {ep:>2} | D {Xc.shape[1]:>5} | {time.time()-t0:.0f}s')
    if br is not None: del Xc
del X

B1 = pd.DataFrame(rows).drop_duplicates('ten', keep='last').set_index('ten')
show = ['nhanh', 'dau_doc', 'mAP_anh', 'mAP_study', 'head', 'medium', 'tail', 'ep', 'D']
print('\n=== B1 · DAU DOC x MISSINGNESS ===')
print(B1[show].to_string(float_format=lambda v: f'{v:.4f}'))
canh_bao_tran(B1)

print(f'\n  (1) PHI TUYEN — MLP - linear, cung nhanh du lieu (cong +0.003):')
for br in ['khong_ehr'] + (['zero', 'median_mask', 'mask_only'] if HAVE_EHR else []):
    a = f'{"cxr" if br == "khong_ehr" else br}_linear'; b = f'{"cxr" if br == "khong_ehr" else br}_mlp'
    if a not in B1.index or b not in B1.index: continue
    d = B1.loc[b, 'mAP_anh'] - B1.loc[a, 'mAP_anh']
    print(f'    {br:<12} {d:+.4f}   {"QUA CONG" if d >= 0.003 else ("trong san" if abs(d) < NOISE else "duoi cong")}')

if HAVE_EHR:
    print(f'\n  (2) MISSINGNESS — lift EHR so voi CXR-only, tach theo dau doc:')
    for hd, suf in (('linear', 'linear'), ('mlp', 'mlp')):
        b0 = B1.loc[f'cxr_{suf}', 'mAP_anh']
        s = '  '.join(f'{br} {B1.loc[f"{br}_{suf}","mAP_anh"]-b0:+.4f}'
                      for br in ('zero', 'median_mask', 'mask_only') if f'{br}_{suf}' in B1.index)
        print(f'    {hd:<7} {s}')
    print('\n    doc: mask_only ~ zero  => EHR chu yeu doc ra LOAI BENH NHAN, khong doc ra BENH.')
    print('         median_mask > zero  => cach dien gia tri co quan trong; 32 C6 dang bo phi.')


=== B1 · DAU DOC x MISSINGNESS ===
                          nhanh dau_doc  mAP_anh  mAP_study   head  medium   tail  ep     D
ten                                                                                        
cxr_linear            khong_ehr  linear   0.2011     0.2121 0.4182  0.1345 0.0412  49   768
cxr_mlp               khong_ehr     mlp   0.2020     0.2165 0.4243  0.1314 0.0418  32   768
zero_linear                zero  linear   0.2048     0.2161 0.4250  0.1383 0.0414  49  1015
zero_mlp                   zero     mlp   0.2049     0.2177 0.4250  0.1337 0.0483  24  1015
median_mask_linear  median_mask  linear   0.2042     0.2150 0.4257  0.1384 0.0383  56  1015
median_mask_mlp     median_mask     mlp   0.2013     0.2139 0.4253  0.1299 0.0405  23  1015
mask_only_linear      mask_only  linear   0.2034     0.2147 0.4231  0.1360 0.0416  49   925
mask_only_mlp         mask_only     mlp   0.2032     0.2161 0.4244  0.1332 0.0436  29   925

  (1) PHI TUYEN — MLP - linear, cung nhanh 

## C8 · **A5.1 · Mốc tương quan** — miễn phí, từ 9 file `p_*.npy` đã có

Đây **không** phải phép thử cho CNN — cả 9 thành viên đều là **RAD-DINO**, khác nhau chỉ ở tiền xử lý
ảnh và biểu diễn. Nên nó là **cận dưới của đa dạng**, và giá trị chính của nó là làm **mốc đối chiếu**
cho `41a`: không có mốc này thì con số `corr` của ConvNeXt không đọc được — `0.70` là cao hay thấp
hoàn toàn phụ thuộc mốc same-backbone nằm ở đâu.

Dùng **trung bình HẠNG** song song với trung bình xác suất: AP là chỉ số **xếp hạng**, và các arm được
huấn luyện riêng nên rất có thể lệch calibration — trung bình xác suất bị arm tự tin nhất kéo đi.

In [9]:
from scipy.stats import rankdata
from itertools import combinations

MEM = {}
for a in ARMS_HAVE:
    for k in ('cls', 'patch', 'both'):
        f = os.path.join(RES_DIR, f'p_{a}_{k}.npy')
        if os.path.exists(f): MEM[f'{a}/{k}'] = np.load(f).astype('float32')
print(f'thanh vien tim duoc: {len(MEM)}')
assert len(MEM) >= 2, 'can it nhat 2 file p_*.npy -> chay 32 C3 truoc'

mAP = lambda p: float(np.nanmean(ap_cols(YM[VA], p)))
S = pd.Series({k: mAP(p) for k, p in MEM.items()}).sort_values(ascending=False)
print('\nmAP anh tung thanh vien:')
print(S.to_string(float_format=lambda v: f'{v:.4f}'))

RK = {k: (rankdata(p, axis=0) / len(p)).astype('float32') for k, p in MEM.items()}   # bat bien calibration

def _mcorr(A, B):
    return float(np.mean([np.corrcoef(A[:, j], B[:, j])[0, 1] for j in range(A.shape[1])]))

pairs = []
for a, b in combinations(MEM, 2):
    pairs.append(dict(cap=f'{a}  vs  {b}',
                      pearson=_mcorr(MEM[a], MEM[b]), spearman=_mcorr(RK[a], RK[b]),
                      cung_arm=a.split('/')[0] == b.split('/')[0],
                      cung_kind=a.split('/')[1] == b.split('/')[1]))
P = pd.DataFrame(pairs).sort_values('spearman')
print('\n=== TUONG QUAN TUNG CAP (TB 40 nhan) ===')
print(P.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

print('\n=== MOC CHO nb41a ===')
_diff_arm  = P[~P.cung_arm & P.cung_kind]           # cung bieu dien, khac tien xu ly anh
_diff_kind = P[P.cung_arm & ~P.cung_kind]           # cung anh, khac bieu dien
print(f'  khac ARM, cung bieu dien   : spearman TB {_diff_arm.spearman.mean():.4f} '
      f'(n={len(_diff_arm)})   <- DA DANG IT NHAT')
print(f'  khac BIEU DIEN, cung arm   : spearman TB {_diff_kind.spearman.mean():.4f} (n={len(_diff_kind)})')
print(f'  toan bo                    : spearman TB {P.spearman.mean():.4f} '
      f'| min {P.spearman.min():.4f} | max {P.spearman.max():.4f}')
MOC = float(_diff_arm.spearman.mean()) if len(_diff_arm) else float(P.spearman.mean())
print(f'\n  => ConvNeXt o 41a phai co spearman THAP HON {MOC:.4f} thi kien truc moi mang lai da dang that.')

def ens(keys, mode='prob'):
    D = MEM if mode == 'prob' else RK
    return np.mean([D[k] for k in keys], 0)

GROUPS = {'3 arm x patch': [k for k in MEM if k.endswith('/patch')],
          'cls + patch (arm chuan)': [k for k in MEM if k.startswith(f'{ARM}/') and not k.endswith('/both')],
          'tat ca': list(MEM)}
er = []
for nm, ks in GROUPS.items():
    if len(ks) < 2: continue
    best = max(S[k] for k in ks)
    for mode in ('prob', 'rank'):
        m = mAP(ens(ks, mode))
        er.append(dict(nhom=nm, n=len(ks), gop=mode, mAP=m, tot_nhat=best, loi=m - best))
ENS = pd.DataFrame(er)
print('\n=== ENSEMBLE (mien phi — khong mot giay GPU nao) ===')
print(ENS.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
_g = ENS.loi.max() if len(ENS) else float('nan')
print(f'\n  loi ich lon nhat {_g:+.4f}  ({"vuot" if _g >= NOISE else "DUOI"} san nhieu {NOISE})')
print('  luu y: 17 (cohort cu) do ensemble 3 checkpoint CUNG mo hinh: macro +0.00112 CI [+0.00027,+0.00197].')
print('         Day cung la truong hop TE NHAT cho ensemble — moi thanh vien deu la RAD-DINO.')
ENS.to_csv(os.path.join(RES35, 'a51_ensemble.csv'), index=False)
P.to_csv(os.path.join(RES35, 'a51_corr.csv'), index=False)

thanh vien tim duoc: 9

mAP anh tung thanh vien:
nosq518/patch   0.2028
lb518/patch     0.2002
lb518/both      0.1974
nosq518/both    0.1974
crop518/patch   0.1917
crop518/both    0.1855
lb518/cls       0.1656
nosq518/cls     0.1651
crop518/cls     0.1638

=== TUONG QUAN TUNG CAP (TB 40 nhan) ===
                             cap  pearson  spearman  cung_arm  cung_kind
  crop518/cls  vs  crop518/patch   0.3692    0.3063      True      False
    crop518/patch  vs  lb518/cls   0.3758    0.3123     False      False
  crop518/patch  vs  nosq518/cls   0.3769    0.3139     False      False
  crop518/cls  vs  nosq518/patch   0.4063    0.3403     False      False
  nosq518/cls  vs  nosq518/patch   0.4160    0.3478      True      False
    lb518/cls  vs  nosq518/patch   0.4152    0.3479     False      False
    crop518/cls  vs  lb518/patch   0.4150    0.3497     False      False
    lb518/patch  vs  nosq518/cls   0.4243    0.3567     False      False
      lb518/cls  vs  lb518/patch   0.4245    

## C9 · Chấm lại **độc lập** — cổng đối chiếu

`ap_cols` tự viết phải khớp `sklearn`. `33` C4 khớp tới `5.06e-09`; giữ đúng mức đó.
Gộp study bằng `groupby` của pandas (đường đi khác hẳn `np.bincount` ở C3) để bắt lỗi lập chỉ mục.

In [10]:
_chk = [f for f in os.listdir(RES35) if f.startswith('p_') and f.endswith('.npy')]
assert _chk, 'chua co du doan nao'
_f = sorted(_chk)[0]; _p = np.load(os.path.join(RES35, _f))

_mine = ap_cols(YM[VA], _p)
_ref  = np.array([average_precision_score(YM[VA][:, j], _p[:, j]) for j in range(len(LAB40))])
_d = float(np.nanmax(np.abs(_mine - _ref)))
print(f'AP anh   : {_f} | lech toi da {_d:.3e}')
assert _d < 1e-6, f'ap_cols lech sklearn {_d:.3e} -> kiem lai cach xu ly diem bang nhau'

# gop study bang duong di KHAC: pandas groupby thay vi bincount
_df = pd.DataFrame(_p, columns=LAB40); _df['sid'] = _sid_va
_isU = V.use.to_numpy()[VA]
_w = np.where((V.view_group.to_numpy()[VA] == 'frontal') & _isU, W_FRONTAL,
     np.where((V.view_group.to_numpy()[VA] == 'lateral') & _isU, W_LATERAL, 0.0))
_df['w'] = _w
_df = _df[_df.w > 0]
_num = _df[LAB40].mul(_df.w, axis=0).groupby(_df.sid).sum()
_den = _df.groupby('sid').w.sum()
_alt = _num.div(_den, axis=0)                       # KHAC C3: khong lay TB trong tung view truoc
_ps, _ok = pool_study(_p)
_common = _alt.index.to_numpy()
_pos = {s: i for i, s in enumerate(_uid)}
_row = np.array([_pos[s] for s in _common])
_nv = np.array([len(np.unique(_df.w[_df.sid == s])) for s in _common[:200]])   # mau 200 study
_same = _nv == 1                                    # study chi mot loai view -> hai cong thuc PHAI trung
_dd = float(np.abs(_alt.to_numpy()[:200][_same] - _ps[_row[:200]][_same]).max())
print(f'gop study: tren {_same.sum()}/200 study MOT loai view, hai cong thuc lech toi da {_dd:.3e}')
assert _dd < 1e-6, 'gop study lech -> kiem lai pool_study'
print('\ncong doi chieu doc lap: DAT')

AP anh   : p_a1_asl20.npy | lech toi da 8.823e-08
gop study: tren 96/200 study MOT loai view, hai cong thuc lech toi da 1.110e-16

cong doi chieu doc lap: DAT


## C10 · Tổng kết + ghi `REPORT_STAGE4_NB35.md`

In ra toàn văn để dán vào repo — theo mẫu `REPORT_frozen_token_fusion_revised.md`, **có** mục
*"những gì kết quả KHÔNG loại trừ"*.

In [11]:
L = []
w = L.append
w('# REPORT · nb35 — Stage 4 sàng lọc trên embedding đã cache')
w('')
w(f'Arm chuẩn `{ARM}/{KIND}` · val {VA.sum():,} ảnh / {len(_uid):,} study · '
  f'lát chọn {SL.sum():,} ảnh cắt từ train theo `subject_id` · sàn nhiễu {NOISE}.')
w('')
w('Mọi epoch dừng và mọi lựa chọn siêu tham số làm trên **lát chọn 15% của train**. '
  'Val chỉ dùng để báo cáo. Test **không bị chạm**.')
w('')

if 'A2' in dir() and len(A2) == 4:
    w('## A2 · Case C')
    w('')
    w('| | gain |'); w('|---|---|')
    w(f'| `gain_cls` | {g_cls:+.4f} |')
    w(f'| `gain_patch` | {g_patch:+.4f} |')
    w(f'| chênh | {g_cls-g_patch:+.4f} |')
    w('')
    w('Cohort cũ: `+0.0081` @cls vs `+0.0003` @patch. `32` C6 đo `+0.0040` @patch trên CXR-LT.')
    w('')

if 'A1' in dir() and len(A1):
    w('## A1 · Loss'); w('')
    w('| loss | mAP ảnh | head | medium | tail | ep* | Δ vs BCE |')
    w('|---|---|---|---|---|---|---|')
    for k, _ in LOSSES:
        if k not in A1.index: continue
        r = A1.loc[k]
        w(f'| `{k}` | {r.mAP_anh:.4f} | {r["head"]:.4f} | {r["medium"]:.4f} | {r["tail"]:.4f} | '
          f'{int(r.ep)} | {r.mAP_anh-base:+.4f} |')
    w(''); w(f'Cổng `+0.003`. Cohort cũ (13 nhãn): ASL − BCE = `+0.0050`, p=0.029.'); w('')

if 'B1' in dir() and len(B1):
    w('## B1 · Đầu đọc × missingness'); w('')
    w('| nhánh | đầu đọc | mAP ảnh | tail | ep* | D |')
    w('|---|---|---|---|---|---|')
    for nm in B1.index:
        r = B1.loc[nm]
        w(f'| {r.nhanh} | {r.dau_doc} | {r.mAP_anh:.4f} | {r["tail"]:.4f} | {int(r.ep)} | {int(r.D)} |')
    w(''); w('Cổng `MLP − linear ≥ +0.003`. Prior: `06` đo FiLM ≈ concat, p=0.308 ở mức CLS.'); w('')

if 'P' in dir() and isinstance(P, pd.DataFrame) and 'spearman' in P:
    w('## A5.1 · Mốc tương quan (cận dưới của đa dạng)'); w('')
    w(f'- khác arm, cùng biểu diễn: spearman TB **{_diff_arm.spearman.mean():.4f}**')
    w(f'- khác biểu diễn, cùng arm: spearman TB **{_diff_kind.spearman.mean():.4f}**')
    w(f'- toàn bộ {len(P)} cặp: **{P.spearman.mean():.4f}** (min {P.spearman.min():.4f})')
    w(f'- lợi ích ensemble lớn nhất: **{_g:+.4f}**')
    w('')
    w(f'**Mốc cho `41a`:** ConvNeXt phải có spearman < **{MOC:.4f}** thì kiến trúc mới mang lại '
      'đa dạng thật. Cả 9 thành viên ở đây đều là RAD-DINO nên đây là **cận dưới**.')
    w('')

w('## Những gì kết quả này KHÔNG loại trừ'); w('')
w('- **Đầu đọc.** Mọi số ở đây là probe trên embedding **đông cứng** đã global-pool. '
  'ML-Decoder đọc đủ 1369 token (`37`) là một lớp hàm khác hẳn; `patch` ở đây là **trung bình**.')
w('- **Cơ chế fusion.** Chỉ concat được thử. `Q=patch, K/V=EHR` (`37`) và adapter trong block '
  '(`39`) chưa ai đo trên cohort này.')
w('- **Backbone.** RAD-DINO đông cứng, chưa fine-tune (`38`).')
w('- **Đích.** Nhãn CXR-LT do NLP trích từ **chính report của phim đó** ⇒ ảnh gần như đủ theo '
  'định nghĩa. Không con số nào ở đây nói gì về tiên lượng, tử vong, chuyển ICU, tái nhập viện.')
w('')

TXT = '\n'.join(L)
open(os.path.join(RES35, 'REPORT_STAGE4_NB35.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)
print(f'\n-> {os.path.join(RES35, "REPORT_STAGE4_NB35.md")}')

# REPORT · nb35 — Stage 4 sàng lọc trên embedding đã cache

Arm chuẩn `nosq518/patch` · val 39,293 ảnh / 23,800 study · lát chọn 39,086 ảnh cắt từ train theo `subject_id` · sàn nhiễu 0.005.

Mọi epoch dừng và mọi lựa chọn siêu tham số làm trên **lát chọn 15% của train**. Val chỉ dùng để báo cáo. Test **không bị chạm**.

## A2 · Case C

| | gain |
|---|---|
| `gain_cls` | +0.0034 |
| `gain_patch` | +0.0038 |
| chênh | -0.0004 |

Cohort cũ: `+0.0081` @cls vs `+0.0003` @patch. `32` C6 đo `+0.0040` @patch trên CXR-LT.

## A1 · Loss

| loss | mAP ảnh | head | medium | tail | ep* | Δ vs BCE |
|---|---|---|---|---|---|---|
| `bce` | 0.2011 | 0.4182 | 0.1345 | 0.0412 | 49 | +0.0000 |
| `asl20` | 0.2061 | 0.4208 | 0.1327 | 0.0594 | 60 | +0.0051 |
| `asl40` | 0.2083 | 0.4195 | 0.1377 | 0.0614 | 49 | +0.0072 |
| `asl41` | 0.2078 | 0.4199 | 0.1368 | 0.0603 | 55 | +0.0067 |
| `wasl` | 0.2079 | 0.4200 | 0.1369 | 0.0605 | 55 | +0.0068 |

Cổng `+0.003`. Cohort cũ (13 nhãn): ASL − BCE = `+0.0050`, p=0.

In [12]:
# ============================================================================
# C6-bis · A1 keo dai ngan sach + seed thu hai
#   Vi sao: `asl20` dung DUNG o tran 60 epoch (tail con dang len) nen so cua no la CAN DUOI.
#   Chi keo dai mot minh no thi thanh so ngan sach — phai keo dai CA BA cung luc.
#   LUU Y: day KHONG phai "huan luyen tiep". CosineAnnealingLR trai tren `epochs`, nen
#   150 epoch la mot duong LR KHAC HAN chu khong phai 60 epoch roi chay them 90.
#   Seed thu hai de biet chenh 0.0022 giua cac bien the ASL co vuot noi dao dong seed khong.
# ============================================================================
EP2   = 150
SEEDS = (42, 43)
KEYS  = ['bce', 'asl20', 'asl40']          # bce lam neo; wasl bo (cham san 15/40 -> thoai hoa ve ASL)

P_LONG = os.path.join(RES35, 'a1_loss_long.csv')
rows = [] if not os.path.exists(P_LONG) else pd.read_csv(P_LONG).to_dict('records')
_done = {(r['ten'], r['seed']) for r in rows}

X = feats(ARM, KIND)
for sd in SEEDS:
    for key in KEYS:
        if (key, sd) in _done: continue
        t0 = time.time()
        p, ep, cur = fit_head(X, YM, head='linear', loss=key, epochs=EP2, seed=sd)
        cur.to_csv(os.path.join(RES35, f'curve_a1long_{key}_s{sd}.csv'), index=False)
        np.save(os.path.join(RES35, f'p_a1long_{key}_s{sd}.npy'), p)
        rows.append(row_of(key, p, ep=ep, seed=sd))
        pd.DataFrame(rows).to_csv(P_LONG, index=False)
        print(f'  s{sd} {key:<6} mAP anh {rows[-1]["mAP_anh"]:.4f} | tail {rows[-1]["tail"]:.4f} '
              f'| ep* {ep:>3}/{EP2} | {time.time()-t0:.0f}s', flush=True)
del X

LG = pd.DataFrame(rows).drop_duplicates(['ten', 'seed'], keep='last')

# --- (1) con arm nao con cham tran khong ---
_hit = LG[LG.ep >= EP2]
print(f'\n  cham tran {EP2} epoch: {list(zip(_hit.ten, _hit.seed)) or "khong con arm nao"}')
if len(_hit):
    print(f'  -> tail VAN dang len. Moi so cua cac arm do van la CAN DUOI; tang EP2 roi chay lai.')

# --- (2) 60 vs 150, tung seed ---
A1old = pd.read_csv(os.path.join(RES35, 'a1_loss.csv')).drop_duplicates('ten', keep='last').set_index('ten')
print(f'\n=== NGAN SACH 60 (seed 42) vs {EP2} ===')
cmp = []
for key in KEYS:
    r = dict(loss=key, e60_anh=A1old.loc[key, 'mAP_anh'], e60_tail=A1old.loc[key, 'tail'],
             e60_ep=int(A1old.loc[key, 'ep']))
    for sd in SEEDS:
        m = LG[(LG.ten == key) & (LG.seed == sd)]
        if len(m):
            r[f'e{EP2}_anh_s{sd}']  = float(m.mAP_anh.iloc[0])
            r[f'e{EP2}_tail_s{sd}'] = float(m['tail'].iloc[0])
            r[f'e{EP2}_ep_s{sd}']   = int(m.ep.iloc[0])
    cmp.append(r)
CMP = pd.DataFrame(cmp).set_index('loss')
print(CMP.to_string(float_format=lambda v: f'{v:.4f}'))

# --- (3) chenh so voi BCE, TRUNG BINH hai seed, kem bien do seed ---
print(f'\n=== CHENH so voi BCE @ {EP2} epoch (TB {len(SEEDS)} seed) ===')
piv = LG.pivot_table(index='ten', columns='seed', values=['mAP_anh', 'tail'])
b_a = piv['mAP_anh'].loc['bce']; b_t = piv['tail'].loc['bce']
for key in KEYS:
    if key == 'bce': continue
    da = (piv['mAP_anh'].loc[key] - b_a); dt = (piv['tail'].loc[key] - b_t)
    print(f'  {key:<6} anh {da.mean():+.4f} (seed spread {da.max()-da.min():.4f}) | '
          f'tail {dt.mean():+.4f} (spread {dt.max()-dt.min():.4f})   '
          f'{"QUA CONG" if da.mean() >= 0.003 else "duoi cong"}')

# --- (4) cau hoi chinh: asl40 co THAT SU hon asl20 khong ---
d = (piv['mAP_anh'].loc['asl40'] - piv['mAP_anh'].loc['asl20'])
dt = (piv['tail'].loc['asl40'] - piv['tail'].loc['asl20'])
spread = float(d.max() - d.min())
print(f'\n=== asl40 - asl20 ===')
print(f'  anh  {d.mean():+.4f}  (tung seed: ' + ', '.join(f'{v:+.4f}' for v in d) + f', bien do {spread:.4f})')
print(f'  tail {dt.mean():+.4f}  (tung seed: ' + ', '.join(f'{v:+.4f}' for v in dt) + ')')
_nhat_quan = bool((d > 0).all() or (d < 0).all())
if abs(d.mean()) < spread:
    print(f'  -> KHONG PHAN DINH: chenh trung binh {abs(d.mean()):.4f} NHO HON bien do giua seed {spread:.4f}.')
    print('     Lay con nao cung duoc; dung phat bieu con nay hon con kia.')
elif abs(d.mean()) < NOISE:
    print(f'  -> nhat quan ve dau ({"asl40" if d.mean() > 0 else "asl20"} luon thang) nhung do lon '
          f'{abs(d.mean()):.4f} duoi san nhieu {NOISE}. Chon theo TAIL neu can mot con.')
else:
    print(f'  -> {"asl40" if d.mean() > 0 else "asl20"} thang that, vuot ca san nhieu lan bien do seed.')

# --- (5) duong cong: dinh o dau, va co truot sau dinh khong ---
print(f'\n  TAIL tren lat chon (dinh -> ep{EP2}):')
for sd in SEEDS:
    for key in KEYS:
        f = os.path.join(RES35, f'curve_a1long_{key}_s{sd}.csv')
        if not os.path.exists(f): continue
        c = pd.read_csv(f); i = int(c['tail'].idxmax())
        print(f'    s{sd} {key:<6} dinh {c["tail"].iloc[i]:.4f}@ep{int(c.ep.iloc[i]):>3} -> '
              f'ep{EP2} {c["tail"].iloc[-1]:.4f}  ({c["tail"].iloc[-1]-c["tail"].iloc[i]:+.4f})')


  s42 bce    mAP anh 0.2081 | tail 0.0597 | ep* 139/150 | 50s
  s42 asl20  mAP anh 0.2015 | tail 0.0605 | ep*  69/150 | 53s
  s42 asl40  mAP anh 0.2058 | tail 0.0570 | ep* 114/150 | 53s
  s43 bce    mAP anh 0.2073 | tail 0.0585 | ep* 110/150 | 45s
  s43 asl20  mAP anh 0.2064 | tail 0.0586 | ep* 101/150 | 53s
  s43 asl40  mAP anh 0.2043 | tail 0.0556 | ep* 103/150 | 53s

  cham tran 150 epoch: khong con arm nao

=== NGAN SACH 60 (seed 42) vs 150 ===
       e60_anh  e60_tail  e60_ep  e150_anh_s42  e150_tail_s42  e150_ep_s42  e150_anh_s43  e150_tail_s43  e150_ep_s43
loss                                                                                                                
bce     0.2011    0.0412      49        0.2081         0.0597          139        0.2073         0.0585          110
asl20   0.2061    0.0594      60        0.2015         0.0605           69        0.2064         0.0586          101
asl40   0.2083    0.0614      49        0.2058         0.0570          114     

In [13]:
# C6-ter · Thang ngan sach — tim cho DUONG COVERGE THAT SU PHANG
#   `ep*` phang KHONG chung minh hoi tu khi CosineAnnealingLR trai dung tren ngan sach:
#   LR chet dung luc do. Bang chung duy nhat la TANG ngan sach ma ket qua khong len nua.
BUDGETS = [300, 600]
P_LAD = os.path.join(RES35, 'a1_budget_ladder.csv')
rows = [] if not os.path.exists(P_LAD) else pd.read_csv(P_LAD).to_dict('records')
_done = {(r['ep_budget'], r['seed']) for r in rows}

X = feats(ARM, KIND)
for E in BUDGETS:
    for sd in (42, 43):
        if (E, sd) in _done: continue
        t0 = time.time()
        p, ep, cur = fit_head(X, YM, head='linear', loss='bce', epochs=E, seed=sd)
        rows.append(dict(ep_budget=E, seed=sd, **{k: v for k, v in row_of('bce', p, ep=ep).items()
                                                  if not k.startswith('ap_')}))
        pd.DataFrame(rows).to_csv(P_LAD, index=False)
        print(f'  E={E:>4} s{sd}  mAP anh {rows[-1]["mAP_anh"]:.4f} | tail {rows[-1]["tail"]:.4f} '
              f'| ep* {ep:>3}/{E} ({ep/E:.0%}) | {time.time()-t0:.0f}s', flush=True)
del X

LAD = pd.DataFrame(rows).drop_duplicates(['ep_budget', 'seed'], keep='last')
_old = [dict(ep_budget=60, seed=42, mAP_anh=0.2011, tail=0.0412, ep=49),
        dict(ep_budget=150, seed=42, mAP_anh=0.2081, tail=0.0597, ep=139),
        dict(ep_budget=150, seed=43, mAP_anh=0.2073, tail=0.0585, ep=110)]
ALL = pd.concat([pd.DataFrame(_old), LAD], ignore_index=True)
G = ALL.groupby('ep_budget').agg(n=('seed','size'), mAP=('mAP_anh','mean'),
                                 tail=('tail','mean'), ep_pct=('ep', lambda s: (s/s.name).mean()
                                 if False else np.nan))
G['ep_tb'] = ALL.groupby('ep_budget').ep.mean()
G['ti_le_ep'] = G.ep_tb / G.index
print('\n=== THANG NGAN SACH (BCE) ===')
print(G.to_string(float_format=lambda v: f'{v:.4f}'))
d = G.mAP.diff()
print('\n  moi nac them duoc:')
for b, v in d.dropna().items():
    print(f'    -> {int(b):>4} epoch: {v:+.4f}  {"CON LEN" if v > 0.003 else "DA PHANG"}')
print('\n  `ti_le_ep` giu nguyen qua cac nac => LICH van dang chan, chua hoi tu.')
print('  `ti_le_ep` GIAM dan               => bat dau cham tran that su.')


  E= 300 s42  mAP anh 0.2067 | tail 0.0591 | ep* 191/300 (64%) | 86s
  E= 300 s43  mAP anh 0.2057 | tail 0.0561 | ep* 205/300 (68%) | 86s
  E= 600 s42  mAP anh 0.2027 | tail 0.0548 | ep* 190/600 (32%) | 168s
  E= 600 s43  mAP anh 0.1994 | tail 0.0491 | ep* 200/600 (33%) | 168s

=== THANG NGAN SACH (BCE) ===
           n    mAP   tail  ep_pct    ep_tb  ti_le_ep
ep_budget                                            
60         1 0.2011 0.0412     NaN  49.0000    0.8167
150        2 0.2077 0.0591     NaN 124.5000    0.8300
300        2 0.2062 0.0576     NaN 198.0000    0.6600
600        2 0.2010 0.0520     NaN 195.0000    0.3250

  moi nac them duoc:
    ->  150 epoch: +0.0066  CON LEN
    ->  300 epoch: -0.0015  DA PHANG
    ->  600 epoch: -0.0051  DA PHANG

  `ti_le_ep` giu nguyen qua cac nac => LICH van dang chan, chua hoi tu.
  `ti_le_ep` GIAM dan               => bat dau cham tran that su.


In [18]:
# ============================================================================
# C6-quinquies · Moi loss o NGAN SACH TOI UU CUA CHINH NO
#   Co dinh E la thien vi: E=60 thien ASL, E=150 thien BCE. Cot `ep*` cho thay `bce_pw05`
#   dinh o 41-42% ngan sach => bi phat y het asl40, checkpoint chon ra chua duoc u.
#   Quet E cho tung loss, CHON E tren LAT CHON (sel-tail), bao cao tren val.
#   Ghi de moi cell A1 truoc do: mot so do ten, mot bo curve, mot bang.
# ============================================================================
BUDGETS, SEEDS = [60, 150, 300], (42, 43)
_N  = int(TF.sum())
_pw = lambda a: torch.as_tensor((((_N - N_POS_TF) / N_POS_TF) ** a).astype('float32')).to(dev)
ARMS_L = {'bce':      lambda: nn.BCEWithLogitsLoss(),
          'bce_pw05': lambda: nn.BCEWithLogitsLoss(pos_weight=_pw(0.5)),
          'bce_pw10': lambda: nn.BCEWithLogitsLoss(pos_weight=_pw(1.0)),
          'asl40':    lambda: ASL(gn=4, gp=0)}

P_E = os.path.join(RES35, 'a1_perloss_budget.csv')
rows = [] if not os.path.exists(P_E) else pd.read_csv(P_E).to_dict('records')
_done = {(r['ten'], r['E'], r['seed']) for r in rows}

X = feats(ARM, KIND)
for E in BUDGETS:                                   # nac nho truoc
    for key, mk in ARMS_L.items():
        for sd in SEEDS:
            if (key, E, sd) in _done: continue
            t0 = time.time()
            p, ep, cur = fit_head(X, YM, head='linear', crit=mk(), epochs=E, seed=sd)
            cur.to_csv(os.path.join(RES35, f'curve_E_{key}_E{E}_s{sd}.csv'), index=False)
            rows.append(dict(E=E, seed=sd, sel_tail=float(cur['tail'].max()),
                             **row_of(key, p, ep=ep)))
            pd.DataFrame(rows).to_csv(P_E, index=False)
            print(f'  E={E:>3} s{sd} {key:<9} val anh {rows[-1]["mAP_anh"]:.4f} '
                  f'| val tail {rows[-1]["tail"]:.4f} | LAT CHON tail {rows[-1]["sel_tail"]:.4f} '
                  f'| ep* {ep:>3} ({ep/E:.0%}) | {time.time()-t0:.0f}s', flush=True)
del X

EB = pd.DataFrame(rows).drop_duplicates(['ten', 'E', 'seed'], keep='last')

print('\n=== TOAN BANG (TB 2 seed) ===')
for col, lab in (('sel_tail', 'LAT CHON tail  <- chon E bang cot NAY'),
                 ('mAP_anh', 'val mAP anh'), ('tail', 'val tail')):
    print(f'\n{lab}:')
    print(EB.pivot_table(index='ten', columns='E', values=col)
            .to_string(float_format=lambda v: f'{v:.4f}'))
print('\nep* / E  (dinh o dau trong ngan sach):')
print((EB.pivot_table(index='ten', columns='E', values='ep')
       / pd.Series(BUDGETS, index=BUDGETS)).to_string(float_format=lambda v: f'{v:.0%}'))

# --- chon E cho tung (loss, seed) bang LAT CHON, roi doc val tai do ---
best = EB.loc[EB.groupby(['ten', 'seed']).sel_tail.idxmax()]
print('\n=== MOI LOSS O NGAN SACH TOI UU CUA CHINH NO ===')
print(best[['ten', 'seed', 'E', 'ep', 'sel_tail', 'mAP_anh', 'tail']]
        .sort_values(['ten', 'seed']).to_string(index=False, float_format=lambda v: f'{v:.4f}'))

agg = best.groupby('ten').agg(E_s42=('E', 'first'), E_s43=('E', 'last'),
                              anh=('mAP_anh', 'mean'), tail=('tail', 'mean'))
agg['anh_spread']  = best.groupby('ten').mAP_anh.agg(lambda s: s.max()-s.min())
agg['tail_spread'] = best.groupby('ten')['tail'].agg(lambda s: s.max()-s.min())
agg['d_anh']  = agg.anh  - agg.loc['bce', 'anh']
agg['d_tail'] = agg['tail'] - agg.loc['bce', 'tail']
print('\n=== BANG KET LUAN ===')
print(agg.sort_values('tail', ascending=False).to_string(float_format=lambda v: f'{v:+.4f}'))
print(f'\n  DOC: chi tin mot chenh lech khi no lon hon CA `*_spread` LAN san nhieu {NOISE}.')
_kd = agg[agg.E_s42 != agg.E_s43]
if len(_kd):
    print(f'  !! hai seed chon E KHAC NHAU cho: {list(_kd.index)} -> E cua chung chua on dinh.')
_hit = EB[(EB.E == max(BUDGETS)) & (EB.ep >= max(BUDGETS))]
print(f'  cham tran {max(BUDGETS)}: {list(zip(_hit.ten, _hit.seed)) or "khong con arm nao"}')


  E= 60 s42 bce       val anh 0.2011 | val tail 0.0412 | LAT CHON tail 0.0324 | ep*  49 (82%) | 20s
  E= 60 s43 bce       val anh 0.2006 | val tail 0.0392 | LAT CHON tail 0.0303 | ep*  56 (93%) | 20s
  E= 60 s42 bce_pw05  val anh 0.2072 | val tail 0.0577 | LAT CHON tail 0.0502 | ep*  46 (77%) | 20s
  E= 60 s43 bce_pw05  val anh 0.2070 | val tail 0.0588 | LAT CHON tail 0.0507 | ep*  45 (75%) | 20s
  E= 60 s42 bce_pw10  val anh 0.2009 | val tail 0.0480 | LAT CHON tail 0.0405 | ep*  36 (60%) | 20s
  E= 60 s43 bce_pw10  val anh 0.2013 | val tail 0.0469 | LAT CHON tail 0.0408 | ep*  46 (77%) | 20s
  E= 60 s42 asl40     val anh 0.2083 | val tail 0.0614 | LAT CHON tail 0.0560 | ep*  49 (82%) | 23s
  E= 60 s43 asl40     val anh 0.2088 | val tail 0.0624 | LAT CHON tail 0.0559 | ep*  55 (92%) | 23s
  E=150 s42 bce       val anh 0.2081 | val tail 0.0597 | LAT CHON tail 0.0526 | ep* 139 (93%) | 44s
  E=150 s43 bce       val anh 0.2073 | val tail 0.0585 | LAT CHON tail 0.0530 | ep* 110 (73%) | 45s


In [19]:
# ============================================================================
# Chan doan · `sel_tail` (max qua epoch) co dang chon sai epoch khong?
#   Max qua NHIEU epoch hon thi lon hon ke ca khi phan phoi khong doi -> so max-cua-300-rut
#   voi max-cua-60-rut la khong hop le. `macro` on dinh hon `tail` hang bac.
#   Doc tu curve da luu; KHONG chay lai gi.
# ============================================================================
import glob, re
rs = []
for f in glob.glob(os.path.join(RES35, 'curve_E_*.csv')):
    m = re.match(r'curve_E_(.+)_E(\d+)_s(\d+)\.csv$', os.path.basename(f))
    if not m: continue
    key, E, sd = m.group(1), int(m.group(2)), int(m.group(3))
    c = pd.read_csv(f)
    i_t, i_m = int(c['tail'].idxmax()), int(c['macro'].idxmax())
    lr = lambda ep: 3e-3 * 0.5 * (1 + np.cos(np.pi * ep / E))
    rs.append(dict(ten=key, E=E, seed=sd,
                   ep_theo_tail=int(c.ep[i_t]), pct_tail=c.ep[i_t]/E, lr_tail=lr(c.ep[i_t]),
                   ep_theo_macro=int(c.ep[i_m]), pct_macro=c.ep[i_m]/E, lr_macro=lr(c.ep[i_m]),
                   tail_tai_macro=float(c['tail'][i_m]), tail_max=float(c['tail'][i_t]),
                   macro_tai_tail=float(c['macro'][i_t]), macro_max=float(c['macro'][i_m])))
D = pd.DataFrame(rs).sort_values(['ten', 'E', 'seed'])
D['tail_mat'] = D.tail_max - D.tail_tai_macro          # gia phai tra khi chon theo macro
print('=== CHON THEO TAIL vs THEO MACRO (tren LAT CHON) ===')
print(D[['ten','E','seed','ep_theo_tail','pct_tail','lr_tail','ep_theo_macro','pct_macro',
         'lr_macro','tail_mat']]
      .to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print(f'\n  chon theo TAIL  : LR trung binh tai diem chon {D.lr_tail.mean():.2e} '
      f'| vi tri {D.pct_tail.mean():.0%} ngan sach | lech chuan vi tri {D.pct_tail.std():.0%}')
print(f'  chon theo MACRO : LR trung binh tai diem chon {D.lr_macro.mean():.2e} '
      f'| vi tri {D.pct_macro.mean():.0%} ngan sach | lech chuan vi tri {D.pct_macro.std():.0%}')
print(f'  gia phai tra tren tail khi chon bang macro: TB {D.tail_mat.mean():+.4f} '
      f'| xau nhat {D.tail_mat.max():+.4f}')
print('\n  DOC: neu `pct_macro` on dinh hon `pct_tail` va `lr_macro` nho hon dang ke thi'
      '\n       chon theo macro cho checkpoint DA U, va `tail_mat` la gia phai tra cho no.')
print('  So cua bang truoc phai doc lai: E duoc chon boi mot dai luong tang theo so lan rut.')


=== CHON THEO TAIL vs THEO MACRO (tren LAT CHON) ===
     ten   E  seed  ep_theo_tail  pct_tail  lr_tail  ep_theo_macro  pct_macro  lr_macro  tail_mat
   asl40  60    42            49    0.8167   0.0002             58     0.9667    0.0000    0.0005
   asl40  60    43            55    0.9167   0.0001             57     0.9500    0.0000    0.0001
   asl40 150    42           114    0.7600   0.0004            136     0.9067    0.0001    0.0011
   asl40 150    43           103    0.6867   0.0007            143     0.9533    0.0000    0.0006
   asl40 300    42           172    0.5733   0.0012            248     0.8267    0.0002    0.0019
   asl40 300    43            30    0.1000   0.0029            255     0.8500    0.0002    0.0015
     bce  60    42            49    0.8167   0.0002             60     1.0000    0.0000    0.0002
     bce  60    43            56    0.9333   0.0000             60     1.0000    0.0000    0.0000
     bce 150    42           139    0.9267   0.0000            13

In [20]:
# ============================================================================
# A2 + B1 chay lai tren cau hinh DA CHOT: asl40, E=60, chon checkpoint theo MACRO.
#   Ban cu chay tren BCE @ E=60 — nen chua hoi tu (bce can 150).
#   Ghi file RIENG, khong de len ban cu, de con doi chieu duoc.
# ============================================================================
E_FIX, CRIT = 60, lambda: ASL(gn=4, gp=0)

def chay_lai(ten_file, jobs):
    P = os.path.join(RES35, ten_file)
    rows = pd.read_csv(P).to_dict('records') if os.path.exists(P) else []
    done = {r['ten'] for r in rows}
    for nm, hd, Xc in jobs:
        if nm in done: continue
        t0 = time.time()
        p, ep, cur = fit_head(Xc, YM, head=hd, crit=CRIT(), epochs=E_FIX)
        np.save(os.path.join(RES35, f'p_v2_{nm}.npy'), p)
        rows.append(dict(**row_of(nm, p, ep=ep), D=Xc.shape[1]))
        pd.DataFrame(rows).to_csv(P, index=False)
        print(f'  {nm:<18} anh {rows[-1]["mAP_anh"]:.4f} | study {rows[-1]["mAP_study"]:.4f} '
              f'| tail {rows[-1]["tail"]:.4f} | ep* {ep:>2}/{E_FIX} | {time.time()-t0:.0f}s', flush=True)
    return pd.DataFrame(rows).drop_duplicates('ten', keep='last').set_index('ten')

# --- A2 ---
J = []
for kd in ('cls', 'patch'):
    Xk = feats(ARM, kd)
    J += [(f'{kd}_cxr', 'linear', Xk), (f'{kd}_cxr_ehr', 'linear', np.concatenate([Xk, EHR['zero']], 1))]
A2v = chay_lai('a2_case_c_v2.csv', J); del J, Xk

gc_, gp_ = (A2v.loc['cls_cxr_ehr','mAP_anh'] - A2v.loc['cls_cxr','mAP_anh'],
            A2v.loc['patch_cxr_ehr','mAP_anh'] - A2v.loc['patch_cxr','mAP_anh'])
print(f'\n=== A2 v2 (asl40, E={E_FIX}, chon theo macro) ===')
print(A2v[['mAP_anh','mAP_study','head','medium','tail','ep','D']]
        .to_string(float_format=lambda v: f'{v:.4f}'))
print(f'  gain_cls {gc_:+.4f} | gain_patch {gp_:+.4f} | TI SO {gp_/gc_:+.3f}'
      f'   (ban BCE@60: +0.0034 / +0.0038 / +1.120)')

# --- B1 ---
Xp = feats(ARM, KIND)
J = [('cxr_linear','linear',Xp), ('cxr_mlp','mlp',Xp)]
for br in ('zero','median_mask','mask_only'):
    Xb = np.concatenate([Xp, EHR[br]], 1)
    J += [(f'{br}_linear','linear',Xb), (f'{br}_mlp','mlp',Xb)]
B1v = chay_lai('b1_head_v2.csv', J); del J, Xp, Xb
print(f'\n=== B1 v2 · MLP - linear ===')
for br, k in (('khong_ehr','cxr'),('zero','zero'),('median_mask','median_mask'),('mask_only','mask_only')):
    d = B1v.loc[f'{k}_mlp','mAP_anh'] - B1v.loc[f'{k}_linear','mAP_anh']
    print(f'  {br:<12} {d:+.4f}   {"QUA CONG" if d >= 0.003 else "trong san"}')
print('  lift EHR (linear): ' + '  '.join(
    f'{br} {B1v.loc[f"{br}_linear","mAP_anh"]-B1v.loc["cxr_linear","mAP_anh"]:+.4f}'
    for br in ('zero','median_mask','mask_only')))


  cls_cxr            anh 0.1884 | study 0.2026 | tail 0.0355 | ep* 53/60 | 23s
  cls_cxr_ehr        anh 0.1852 | study 0.1985 | tail 0.0315 | ep* 25/60 | 24s
  patch_cxr          anh 0.2083 | study 0.2225 | tail 0.0614 | ep* 49/60 | 23s
  patch_cxr_ehr      anh 0.2079 | study 0.2204 | tail 0.0573 | ep* 46/60 | 24s

=== A2 v2 (asl40, E=60, chon theo macro) ===
               mAP_anh  mAP_study   head  medium   tail  ep     D
ten                                                              
cls_cxr         0.1884     0.2026 0.4057  0.1169 0.0355  53   768
cls_cxr_ehr     0.1852     0.1985 0.4062  0.1114 0.0315  25  1015
patch_cxr       0.2083     0.2225 0.4195  0.1377 0.0614  49   768
patch_cxr_ehr   0.2079     0.2204 0.4219  0.1375 0.0573  46  1015
  gain_cls -0.0032 | gain_patch -0.0004 | TI SO +0.130   (ban BCE@60: +0.0034 / +0.0038 / +1.120)
  cxr_linear         anh 0.2083 | study 0.2225 | tail 0.0614 | ep* 49/60 | 23s
  cxr_mlp            anh 0.1978 | study 0.2088 | tail 0.0348 | ep

In [21]:
import re, collections
S = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
fam = lambda c: ('mask' if c.endswith('_mask') else
                 'dh_lab' if c.startswith('dh_lab_') else
                 'n_lab'  if c.startswith('n_lab_')  else
                 'ctx'    if c.startswith('ctx_')    else
                 'vital_meta' if c.startswith('vital_') else 'khac')
for ten, cols in (('Q_CTX', S['Q_CTX']), ('Q_FULL - Q_CTX', sorted(set(S['Q_FULL'])-set(S['Q_CTX'])))):
    C = collections.Counter(map(fam, cols))
    print(f'\n{ten}  ({len(cols)} cot)')
    for k, v in C.most_common(): print(f'   {k:<12} {v:>4}   vd: ' +
        ', '.join([c for c in cols if fam(c) == k][:3]))



Q_CTX  (157 cot)
   dh_lab         54   vd: dh_lab_albumin_24h, dh_lab_bicarbonate_24h, dh_lab_bilirubin_24h
   n_lab          54   vd: n_lab_albumin_24h, n_lab_bicarbonate_24h, n_lab_bilirubin_24h
   mask           45   vd: lab_albumin_24h_mask, lab_bicarbonate_24h_mask, lab_bilirubin_24h_mask
   ctx             2   vd: ctx_is_inpatient, ctx_is_ed
   vital_meta      2   vd: vital_n_meas, vital_delta_h

Q_FULL - Q_CTX  (90 cot)
   khac           84   vd: age, como_cancer, como_chf
   vital_meta      6   vd: vital_dbp, vital_hr, vital_rr


In [23]:
# ============================================================================
# Bo early-stop trong mot lan chay: lay mo hinh o EPOCH CUOI (cosine ve 0 => luon u het).
#   Ly do: chon epoch tren lat cat van rung. Seed 44 chon ep34/60 cho CA BON arm,
#   LR tai do 1.19e-3 (gap 23x ep55), lam bien do seed nhay tu 0.001 len 0.007 —
#   lon hon moi hieu ung dang do. Chan doan cho thay macro von da chon 93%+-5% ngan sach,
#   nen lay hang epoch cuoi gan nhu khong mat gi ma bo duoc ca mot lop loi.
#   `E` van la sieu tham so, van chon tren lat cat.
# ============================================================================
def fit_cuoi(X, y, *, head='linear', crit=None, epochs=60, lr=3e-3, wd=1e-4, bs=4096,
             hid=None, drop=0.2, seed=SEED):
    torch.manual_seed(seed)
    mu, sd = X[TF].mean(0, keepdims=True), X[TF].std(0, keepdims=True) + 1e-6
    z = lambda m: torch.from_numpy(((X[m] - mu) / sd).astype('float32')).to(dev)
    xt, xs, xv = z(TF), z(SL), z(VA)
    yt = torch.from_numpy(y[TF]).to(dev); ys = y[SL]
    hid = min(1024, 4*xt.shape[1]) if hid is None else hid
    m = (nn.Linear(xt.shape[1], y.shape[1]) if head == 'linear'
         else MLPHead(xt.shape[1], y.shape[1], hid, drop)).to(dev)
    crit = (crit if crit is not None else nn.BCEWithLogitsLoss()).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    bs = int(min(bs, max(256, len(xt) // 64))); n = len(xt); curve = []
    for ep in range(epochs):
        m.train(); perm = torch.randperm(n, device=dev)
        for i in range(0, n, bs):
            ix = perm[i:i+bs]; opt.zero_grad(set_to_none=True)
            crit(m(xt[ix]), yt[ix]).backward(); opt.step()
        sch.step()
    m.eval()                                   # KHONG chon epoch — lay thang epoch cuoi
    with torch.no_grad():
        sel = ap_cols(ys, torch.sigmoid(m(xs)).cpu().numpy())     # diem tren LAT CHON, de chon E
        pv  = torch.sigmoid(m(xv)).cpu().numpy()
    del xt, xs, xv, yt
    if dev == 'cuda': torch.cuda.empty_cache()
    return pv.astype('float32'), float(np.nanmean(sel)), tiers(sel)

E_FIX, SEEDS = 60, (42, 43, 44)
P = os.path.join(RES35, 'ehr_featset_cuoi.csv')
rows = pd.read_csv(P).to_dict('records') if os.path.exists(P) else []
done = {(r['ten'], r['seed']) for r in rows}

Xp = feats(ARM, KIND)
for nm, br in [('cxr', None), ('zero', 'zero'), ('median_mask', 'median_mask'), ('mask_only', 'mask_only')]:
    Xc = Xp if br is None else np.concatenate([Xp, EHR[br]], 1)
    for sd in SEEDS:
        if (nm, sd) in done: continue
        t0 = time.time()
        p, sel_macro, sel_t = fit_cuoi(Xc, YM, crit=ASL(gn=4, gp=0), epochs=E_FIX, seed=sd)
        rows.append(dict(seed=sd, D=Xc.shape[1], sel_macro=sel_macro, sel_tail=sel_t['tail'],
                         **{k: v for k, v in row_of(nm, p).items() if not k.startswith('ap_')}))
        pd.DataFrame(rows).to_csv(P, index=False)
        print(f'  s{sd} {nm:<12} anh {rows[-1]["mAP_anh"]:.4f} | study {rows[-1]["mAP_study"]:.4f} '
              f'| tail {rows[-1]["tail"]:.4f} | {time.time()-t0:.0f}s', flush=True)
    if br is not None: del Xc
del Xp

F = pd.DataFrame(rows).drop_duplicates(['ten', 'seed'], keep='last')
S = F.pivot_table(index='ten', columns='seed', values=['mAP_anh', 'mAP_study', 'tail'])
tb = pd.DataFrame({c: S[c].mean(1) for c in ('mAP_anh', 'mAP_study', 'tail')})
for c in ('mAP_anh', 'mAP_study', 'tail'):
    tb[f'{c}_bd'] = S[c].max(1) - S[c].min(1)
    tb[f'd_{c}']  = tb[c] - tb.loc['cxr', c]
print(f'\n=== EPOCH CUOI — asl40, E={E_FIX}, {len(SEEDS)} seed ===')
print(tb.reindex(['cxr', 'zero', 'median_mask', 'mask_only'])
        .to_string(float_format=lambda v: f'{v:+.4f}'))
print('\n  So sanh voi ban CHON EPOCH (3 seed):  anh_bd 0.0058-0.0072 | tail_bd 0.0039-0.0066')
print(f'  Ban nay:                              anh_bd {tb.mAP_anh_bd.min():.4f}-{tb.mAP_anh_bd.max():.4f}'
      f' | tail_bd {tb.tail_bd.min():.4f}-{tb.tail_bd.max():.4f}')
print('\n  Bien do CO SUP KHONG la cau hoi duy nhat cua cell nay. Neu co, day la giao thuc cho 37.')
print('  Nguong doc: so sanh `d_*` voi `*_bd` cua CHINH chi so do — KHONG dung san 0.005 cho tail.')


  s42 cxr          anh 0.2093 | study 0.2229 | tail 0.0614 | 10s
  s43 cxr          anh 0.2090 | study 0.2226 | tail 0.0624 | 11s
  s44 cxr          anh 0.2097 | study 0.2233 | tail 0.0620 | 10s
  s42 zero         anh 0.2115 | study 0.2237 | tail 0.0592 | 11s
  s43 zero         anh 0.2108 | study 0.2234 | tail 0.0581 | 11s
  s44 zero         anh 0.2117 | study 0.2240 | tail 0.0579 | 11s
  s42 median_mask  anh 0.2112 | study 0.2232 | tail 0.0584 | 11s
  s43 median_mask  anh 0.2105 | study 0.2227 | tail 0.0575 | 12s
  s44 median_mask  anh 0.2113 | study 0.2233 | tail 0.0567 | 13s
  s42 mask_only    anh 0.2113 | study 0.2244 | tail 0.0618 | 12s
  s43 mask_only    anh 0.2110 | study 0.2244 | tail 0.0617 | 12s
  s44 mask_only    anh 0.2109 | study 0.2240 | tail 0.0595 | 13s

=== EPOCH CUOI — asl40, E=60, 3 seed ===
             mAP_anh  mAP_study    tail  mAP_anh_bd  d_mAP_anh  mAP_study_bd  d_mAP_study  tail_bd  d_tail
ten                                                                    

In [24]:
# Dau day cuoi cua A1: `E` toi uu duoi giao thuc MOI (epoch cuoi, khong early-stop).
# Chon E bang `sel_macro` tren lat cat; val chi de doc.
Xp = feats(ARM, KIND); rs = []
for key, mk in (('asl40', lambda: ASL(gn=4, gp=0)), ('bce', lambda: nn.BCEWithLogitsLoss())):
    for E in (30, 60, 100, 150):
        for sd in (42, 43):
            p, sm, st = fit_cuoi(Xp, YM, crit=mk(), epochs=E, seed=sd)
            rs.append(dict(loss=key, E=E, seed=sd, sel_macro=sm, sel_tail=st['tail'],
                           **{k: v for k, v in row_of(key, p).items() if not k.startswith('ap_')}))
            print(f'  {key:<6} E={E:>3} s{sd}  sel_macro {sm:.4f} | anh {rs[-1]["mAP_anh"]:.4f} '
                  f'| tail {rs[-1]["tail"]:.4f}', flush=True)
del Xp
R = pd.DataFrame(rs)
for c in ('sel_macro', 'mAP_anh', 'tail'):
    print(f'\n{c} (TB 2 seed):')
    print(R.pivot_table(index='loss', columns='E', values=c).to_string(float_format=lambda v: f'{v:.4f}'))
print('\n  Chon E theo `sel_macro` (lat cat). Neu no chon cung mot E cho ca hai loss')
print('  thi cau chuyen "asl hoi tu nhanh hon" la san pham cua early-stop, khong phai cua loss.')


  asl40  E= 30 s42  sel_macro 0.1973 | anh 0.2083 | tail 0.0581
  asl40  E= 30 s43  sel_macro 0.1967 | anh 0.2084 | tail 0.0611
  asl40  E= 60 s42  sel_macro 0.1984 | anh 0.2093 | tail 0.0614
  asl40  E= 60 s43  sel_macro 0.1980 | anh 0.2090 | tail 0.0624
  asl40  E=100 s42  sel_macro 0.1979 | anh 0.2087 | tail 0.0588
  asl40  E=100 s43  sel_macro 0.1975 | anh 0.2085 | tail 0.0605
  asl40  E=150 s42  sel_macro 0.1980 | anh 0.2075 | tail 0.0561
  asl40  E=150 s43  sel_macro 0.1987 | anh 0.2081 | tail 0.0578
  bce    E= 30 s42  sel_macro 0.1810 | anh 0.1926 | tail 0.0256
  bce    E= 30 s43  sel_macro 0.1804 | anh 0.1916 | tail 0.0237
  bce    E= 60 s42  sel_macro 0.1898 | anh 0.2012 | tail 0.0405
  bce    E= 60 s43  sel_macro 0.1894 | anh 0.2006 | tail 0.0394
  bce    E=100 s42  sel_macro 0.1947 | anh 0.2066 | tail 0.0539
  bce    E=100 s43  sel_macro 0.1947 | anh 0.2062 | tail 0.0533
  bce    E=150 s42  sel_macro 0.1969 | anh 0.2081 | tail 0.0600
  bce    E=150 s43  sel_macro 0.1971 | a

In [25]:
# ============================================================================
# Tran cua DINH TUYEN THEO BENH — do truoc khi dung B2c.
#   Fusion toan cuc ep MOT danh doi len ca 40 nhan: +0.0018 macro doi lay -0.0035 tail.
#   Neu moi nhan tu chon doc EHR hay khong, tran la mean_j max(AP_cxr_j, AP_ehr_j).
#   Day la ORACLE tren val => CAN TREN lac quan. Neu can tren nho thi B2c khong dang dung.
# ============================================================================
Xp = feats(ARM, KIND)
AP = {}
for nm, br in (('cxr', None), ('mask_only', 'mask_only'), ('zero', 'zero')):
    Xc = Xp if br is None else np.concatenate([Xp, EHR[br]], 1)
    acc = []
    for sd in (42, 43):
        p, _, _ = fit_cuoi(Xc, YM, crit=ASL(gn=4, gp=0), epochs=60, seed=sd)
        acc.append(ap_cols(YM[VA], p))
    AP[nm] = np.mean(acc, 0)
    print(f'  {nm:<10} macro {AP[nm].mean():.4f}', flush=True)
    if br is not None: del Xc
del Xp

TIER_OF = {l: t for t in TIER for l in np.array(LAB40)[TIER[t]]}
for ehr in ('mask_only', 'zero'):
    d  = AP[ehr] - AP['cxr']
    orc = np.maximum(AP['cxr'], AP[ehr])
    T = pd.DataFrame(dict(nhan=LAB40, tier=[TIER_OF[l] for l in LAB40],
                          cxr=AP['cxr'], ehr=AP[ehr], d=d))
    print(f'\n=== {ehr} ===')
    print(f'  fusion toan cuc  {AP[ehr].mean():.4f}   ({d.mean():+.4f})')
    print(f'  ORACLE tung nhan {orc.mean():.4f}   ({orc.mean()-AP["cxr"].mean():+.4f})  <- TRAN cua B2c')
    print(f'  so nhan EHR giup: {(d > 0).sum()}/40  |  hai: {(d < 0).sum()}/40')
    print('  theo tier  (d duong / tong):')
    for t in ('head', 'medium', 'tail'):
        m = T.tier == t
        print(f'    {t:<7} {int((T.d[m] > 0).sum())}/{int(m.sum())}  '
              f'| d TB {T.d[m].mean():+.4f} | oracle - cxr {np.maximum(0, T.d[m]).mean():+.4f}')
    print('\n  8 nhan EHR giup nhat / 8 nhan EHR hai nhat:')
    print(pd.concat([T.nlargest(8, 'd'), T.nsmallest(8, 'd')])
            .to_string(index=False, float_format=lambda v: f'{v:+.4f}'))


  cxr        macro 0.2091
  mask_only  macro 0.2112
  zero       macro 0.2111

=== mask_only ===
  fusion toan cuc  0.2112   (+0.0020)
  ORACLE tung nhan 0.2126   (+0.0034)  <- TRAN cua B2c
  so nhan EHR giup: 24/40  |  hai: 16/40
  theo tier  (d duong / tong):
    head    11/13  | d TB +0.0046 | oracle - cxr +0.0046
    medium  9/16  | d TB +0.0014 | oracle - cxr +0.0028
    tail    4/11  | d TB -0.0001 | oracle - cxr +0.0029

  8 nhan EHR giup nhat / 8 nhan EHR hai nhat:
                  nhan   tier     cxr     ehr       d
     Hydropneumothorax   tail +0.1086 +0.1329 +0.0243
          Rib Fracture medium +0.0838 +0.0991 +0.0152
          Pneumothorax   head +0.4591 +0.4727 +0.0136
              Fracture   head +0.1214 +0.1325 +0.0111
             Pneumonia   head +0.2789 +0.2875 +0.0085
                 Edema   head +0.5147 +0.5216 +0.0069
          Infiltration   head +0.0543 +0.0608 +0.0066
               Fissure medium +0.0577 +0.0642 +0.0065
      Pneumoperitoneum   tail +0.165

In [27]:
# ============================================================================
# NULL cua ORACLE: `max` tren val luon cho loi ich duong ke ca khi hai arm GIONG HET.
#   Dung hai CAP seed cua cung arm `cxr` (chenh lech that = 0) -> loi ich do la THIEN LECH.
#   Phai khop cach uoc luong cua bang that: moi arm la TB cua 2 seed, roi moi lay max.
# ============================================================================
Xp = feats(ARM, KIND)
ap = {}
for sd in (42, 43, 44, 45):
    p, _, _ = fit_cuoi(Xp, YM, crit=ASL(gn=4, gp=0), epochs=60, seed=sd)
    ap[sd] = ap_cols(YM[VA], p)
    print(f'  cxr s{sd}  macro {ap[sd].mean():.4f}', flush=True)
del Xp

A, B = np.mean([ap[42], ap[43]], 0), np.mean([ap[44], ap[45]], 0)   # hai uoc luong DOC LAP
nen  = np.mean([A, B], 0).mean()
null = np.maximum(A, B).mean() - nen
print(f'\n=== NULL ===')
print(f'  hai cap seed cua CUNG arm: nen {nen:.4f} | oracle {np.maximum(A,B).mean():.4f}')
print(f'  THIEN LECH thuan cua phep max: {null:+.4f}')
for ehr, obs in (('mask_only', 0.0034), ('zero', 0.0045)):
    print(f'  {ehr:<10} oracle quan sat {obs:+.4f}  ->  SAU khi tru thien lech {obs-null:+.4f}')
print(f'\n  Fusion toan cuc dang cho +0.0020. Du dia THAT cua B2c = (oracle - null) - 0.0020.')
print(f'  Neu con duong va lon hon bien do seed (~0.0007) thi B2c dang dung.')
print(f'  Neu con <= 0 thi dinh tuyen chi dang do lai nhieu tung nhan -> BO B2c, giu fusion toan cuc.')


  cxr s42  macro 0.2093
  cxr s43  macro 0.2090
  cxr s44  macro 0.2097
  cxr s45  macro 0.2082

=== NULL ===
  hai cap seed cua CUNG arm: nen 0.2090 | oracle 0.2097
  THIEN LECH thuan cua phep max: +0.0006
  mask_only  oracle quan sat +0.0034  ->  SAU khi tru thien lech +0.0028
  zero       oracle quan sat +0.0045  ->  SAU khi tru thien lech +0.0039

  Fusion toan cuc dang cho +0.0020. Du dia THAT cua B2c = (oracle - null) - 0.0020.
  Neu con duong va lon hon bien do seed (~0.0007) thi B2c dang dung.
  Neu con <= 0 thi dinh tuyen chi dang do lai nhieu tung nhan -> BO B2c, giu fusion toan cuc.


In [26]:
# Truoc khi tieu 4-8 gio cho lb1022: DO PHAN GIAI co dau hieu gi trong so da co khong?
# `crop518` phan giai cao hon 1.30x nhung mat 28.5% FOV. Neu phan giai la rang buoc that,
# nhom nhan TIEU DIEM NHO phai chuong crop518 con nhom LAN TOA thi khong.
R = pd.read_csv(os.path.join(RES_DIR, 'cxr_only.csv')).drop_duplicates(['arm','kind'], keep='last')
apc = [c for c in R.columns if c.startswith('ap_')]
g = lambda a: R[R.arm.eq(a) & R.kind.eq('patch')][apc].iloc[0].to_numpy('float64')
NHO = ['Clavicle Fracture','Rib Fracture','Fracture','Nodule','Granuloma','Calcification of the Aorta',
       'Round(ed) Atelectasis','Fissure','Lung Lesion','Pneumomediastinum','Subcutaneous Emphysema']
T = pd.DataFrame(dict(nhan=[c[3:] for c in apc], crop=g('crop518'), lb=g('lb518'), nosq=g('nosq518')))
T['tier']  = T.nhan.map({l: t for t in TIER for l in np.array(LAB40)[TIER[t]]})
T['nhom']  = np.where(T.nhan.isin(NHO), 'tieu diem nho', 'lan toa')
T['d_res'] = T.crop - T.lb          # +: phan giai cao THANG du mat FOV
print(T.groupby('nhom').agg(n=('d_res','size'), d_res_TB=('d_res','mean'),
                            so_duong=('d_res', lambda s: int((s>0).sum())))
        .to_string(float_format=lambda v: f'{v:+.4f}'))
print(f"\ntoan bo 40 nhan: d_res TB {T.d_res.mean():+.4f} | {int((T.d_res>0).sum())}/40 duong")
print('\nDOC: neu `tieu diem nho` co d_res DUONG con `lan toa` AM => phan giai that su rang buoc,')
print('     va lb1022 (giu nguyen FOV, gap doi phan giai) dang gia 4-8 gio.')
print('     Neu ca hai nhom cung AM => mat FOV che het, phep thu nay khong ket luan duoc,')
print('     va lb1022 van la cach duy nhat tach hai bien.')
print(T.sort_values('d_res', ascending=False).head(10).to_string(index=False, float_format=lambda v: f'{v:+.4f}'))



                n  d_res_TB  so_duong
nhom                                 
lan toa        29   -0.0088        12
tieu diem nho  11   -0.0077         1

toan bo 40 nhan: d_res TB -0.0085 | 13/40 duong

DOC: neu `tieu diem nho` co d_res DUONG con `lan toa` AM => phan giai that su rang buoc,
     va lb1022 (giu nguyen FOV, gap doi phan giai) dang gia 4-8 gio.
     Neu ca hai nhom cung AM => mat FOV che het, phep thu nay khong ket luan duoc,
     va lb1022 van la cach duy nhat tach hai bien.
            nhan    crop      lb    nosq   tier          nhom   d_res
     Atelectasis +0.5567 +0.5464 +0.5526   head       lan toa +0.0104
          Hernia +0.5320 +0.5224 +0.5341 medium       lan toa +0.0096
    Pneumothorax +0.4497 +0.4407 +0.4507   head       lan toa +0.0091
    Cardiomegaly +0.6424 +0.6339 +0.6410   head       lan toa +0.0086
    Lung Opacity +0.5361 +0.5283 +0.5341   head       lan toa +0.0077
            Mass +0.1823 +0.1754 +0.1861 medium       lan toa +0.0069
Pleural Effusion